# 27. 時間足再集約とBASE / 50特徴量の再統合比較
出典: FX (2).ipynb、セルindex [56, 57, 58]。保存出力は results/imported_20260909/ を参照。
研究履歴です。実行順・Notebook内変数・元の価格CSVに依存し、エラーが出たコードも保存しています。
自動判定の文言は元実験の判定であり、監査済みの結論ではありません。全セル一括実行は再現手順ではありません。
[USER_HOME] は匿名化した元のパスです。元Notebook内の案内や依頼文は研究資料として保持しています。


## 元セルindex 56


In [ ]:
# ============================================================
# CHAMPION REINTEGRATION TEST
# Formal HGB Champion vs +Regime vs +Volatility+Regime
#
# 目的:
# 1) 正式Championの30特徴量 + 固定HGB設定をコード内で完全固定
# 2) fallback model / fallback feature list は使わない
# 3) Validationだけで Calibration -> Threshold/Session -> Sizing を選ぶ
# 4) 2020-2025 = Development OOS, 2026 = Confirmation
# 5) signal t -> Open(t+1) -> Close(t+2), 30分固定Exit
# 6) 非重複ポジション、round-trip cost = 0.00004
# ============================================================

import math
import warnings
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

warnings.filterwarnings("ignore")


# ============================================================
# 0. 固定設定
# ============================================================

CHAMPION_FEATURES = [
    "return_1", "return_2", "return_4", "return_8", "return_16",
    "vol_4", "vol_8", "vol_16", "vol_32",

    "ma5_distance", "ma5_slope",
    "ma10_distance", "ma10_slope",
    "ma20_distance", "ma20_slope",
    "ma50_distance", "ma50_slope",
    "ma100_distance", "ma100_slope",

    "body",
    "upper_wick",
    "lower_wick",
    "range_pct",

    "rsi14",
    "atr14",

    "distance_high_16",
    "distance_low_16",

    "hour_sin",
    "hour_cos",
    "weekday",
]


# ------------------------------------------------------------
# Volatility追加特徴量
# ------------------------------------------------------------

VOLATILITY_FEATURES = [
    "vol_64",
    "vol_96",

    "vol_ratio_4_32",
    "vol_ratio_8_32",
    "vol_ratio_16_64",

    "range_mean_8",
    "range_mean_32",
    "range_ratio_8_32",

    "range_z_20",
    "range_z_50",

    "abs_return_z_32",
    "abs_return_z_96",

    "gap_abs_1",
]


# ------------------------------------------------------------
# Regime追加特徴量
# ------------------------------------------------------------

REGIME_FEATURES = [
    "adx14",
    "adx28",

    "ma20_50_spread",
    "ma50_100_spread",

    "trend_strength_20",
    "trend_strength_50",

    "price_pos_20",
    "price_pos_50",

    "ma_alignment_score",
    "slope_alignment_score",

    "directional_persistence_16",
]


FEATURE_SETS = {

    "CHAMPION":
        CHAMPION_FEATURES,

    "CHAMPION_PLUS_REGIME":
        CHAMPION_FEATURES
        + REGIME_FEATURES,

    "CHAMPION_PLUS_VOL_REGIME":
        CHAMPION_FEATURES
        + VOLATILITY_FEATURES
        + REGIME_FEATURES,
}


# ------------------------------------------------------------
# 正式HGB Champion設定
# ------------------------------------------------------------

HGB_CONFIG = dict(
    learning_rate=0.05,
    max_iter=250,
    max_leaf_nodes=15,
    min_samples_leaf=30,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42,
)


THRESHOLDS = (
    0.50,
    0.52,
    0.54,
    0.55,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
)


SESSION_POLICIES = (
    "ALL",
    "UTC_13_24",
    "UTC_21_24",
    "EXCLUDE_08_13",
)


SIZING_POLICIES = (
    "FIXED",
    "GENTLE",
    "MODERATE",
    "STRONG",
)


CALIBRATION_METHODS = (
    "RAW",
    "PLATT",
    "ISOTONIC",
)


COST = 0.00004

COST_MULTIPLIERS = (
    1.0,
    1.5,
    2.0,
)


DEVELOPMENT_YEARS = tuple(
    range(2020, 2026)
)

CONFIRMATION_YEAR = 2026


MIN_TRAIN_ROWS = 5000

MIN_EVAL_ROWS = 100

MIN_VALIDATION_TRADES = 100

MIN_PRIOR_YEARS_FOR_OOF = 2


BOOTSTRAP_ITERATIONS = 2000

BOOTSTRAP_BLOCK_DAYS = 10

RANDOM_SEED = 42


print("=" * 100)

print(
    "CHAMPION REINTEGRATION TEST"
)

print("=" * 100)

print(
    "Formal Champion features:",
    len(CHAMPION_FEATURES)
)

print(
    "Regime additions:",
    len(REGIME_FEATURES)
)

print(
    "Volatility additions:",
    len(VOLATILITY_FEATURES)
)

print(
    "HGB config:",
    HGB_CONFIG
)

print(
    "NO FALLBACK MODEL"
)

print(
    "NO FALLBACK BASE FEATURE LIST"
)



# ============================================================
# 1. Notebookから15分足OHLCを取得
# ============================================================

def _get_notebook_namespace():

    try:

        return get_ipython().user_ns

    except Exception:

        return globals()



def _looks_like_15m_ohlc(obj):

    if not isinstance(
        obj,
        pd.DataFrame
    ):

        return False


    if len(obj) < 1000:

        return False


    cols = {
        str(c).strip().lower()
        for c in obj.columns
    }


    if not {
        "open",
        "high",
        "low",
        "close"
    }.issubset(cols):

        return False


    if isinstance(
        obj.index,
        pd.DatetimeIndex
    ):

        idx = obj.index[
            :min(5000, len(obj))
        ]

    else:

        time_col = next(

            (
                c
                for c in obj.columns

                if str(c).strip().lower()

                in {
                    "timestamp",
                    "datetime",
                    "date",
                    "time"
                }
            ),

            None
        )


        if time_col is None:

            return False


        try:

            idx = pd.DatetimeIndex(

                pd.to_datetime(

                    obj[time_col].iloc[
                        :min(
                            5000,
                            len(obj)
                        )
                    ],

                    utc=True
                )
            )

        except Exception:

            return False


    if len(idx) < 20:

        return False


    diffs = pd.Series(
        idx[1:] - idx[:-1]
    )


    diffs = diffs[
        diffs > pd.Timedelta(0)
    ]


    if diffs.empty:

        return False


    return (
        diffs.median()
        ==
        pd.Timedelta(
            minutes=15
        )
    )



def find_bars():

    ns = _get_notebook_namespace()


    preferred = [

        "bars",

        "bars15",

        "bars_15m",

        "bars15m",

        "data15",

        "df15",

        "ohlc15",
    ]


    for name in preferred:

        obj = ns.get(name)


        if _looks_like_15m_ohlc(
            obj
        ):

            print(

                f"[DATA] Using notebook DataFrame: "
                f"{name} ({len(obj):,} rows)"
            )

            return obj.copy()


    candidates = []


    for name, obj in ns.items():

        try:

            if _looks_like_15m_ohlc(
                obj
            ):

                candidates.append(

                    (
                        len(obj),
                        name,
                        obj
                    )
                )

        except Exception:

            pass


    if not candidates:

        raise RuntimeError(

            "15分足OHLC DataFrameが"
            "Notebook内に見つかりません。"
            "\n`bars` という変数名で"
            "15分足OHLCを用意してから"
            "再実行してください。"
        )


    candidates.sort(

        reverse=True,

        key=lambda x: x[0]
    )


    n, name, obj = candidates[0]


    print(

        f"[DATA] Auto-selected DataFrame: "
        f"{name} ({n:,} rows)"
    )


    return obj.copy()



def normalize_bars(
    frame
):

    out = frame.copy()


    rename = {

        c:
        str(c).strip().lower()

        for c in out.columns
    }


    out = out.rename(
        columns=rename
    )


    if not isinstance(
        out.index,
        pd.DatetimeIndex
    ):

        time_col = next(

            (
                c

                for c in out.columns

                if c in {

                    "timestamp",
                    "datetime",
                    "date",
                    "time"
                }
            ),

            None
        )


        if time_col is None:

            raise ValueError(

                "DatetimeIndexまたは"
                "timestamp/datetime/date/time列"
                "が必要です。"
            )


        idx = pd.to_datetime(

            out.pop(
                time_col
            ),

            utc=True,

            errors="coerce"
        )


        out.index = idx


    else:

        out.index = pd.to_datetime(

            out.index,

            utc=True,

            errors="coerce"
        )


    out = out.loc[
        ~out.index.isna()
    ].copy()


    out = out.sort_index()


    needed = [
        "open",
        "high",
        "low",
        "close"
    ]


    missing = [

        c
        for c in needed

        if c not in out.columns
    ]


    if missing:

        raise ValueError(

            f"OHLC列が不足: {missing}"
        )


    out = (

        out[needed]

        .apply(
            pd.to_numeric,
            errors="coerce"
        )

        .dropna()
    )


    if out.index.has_duplicates:

        dup = out.loc[

            out.index.duplicated(
                keep=False
            )
        ]


        bad = []


        for ts, g in dup.groupby(
            level=0
        ):

            if len(
                g.drop_duplicates()
            ) > 1:

                bad.append(
                    ts
                )


        if bad:

            raise ValueError(

                "同一timestampに異なる"
                f"OHLCがあります: {bad[:3]}"
            )


        out = out[
            ~out.index.duplicated(
                keep="first"
            )
        ]


    if (
        out <= 0
    ).any().any():

        raise ValueError(
            "OHLCに0以下の値があります。"
        )


    if (

        out["high"]

        <

        out[
            [
                "open",
                "close",
                "low"
            ]
        ].max(axis=1)

    ).any():

        raise ValueError(
            "Highの整合性エラーがあります。"
        )


    if (

        out["low"]

        >

        out[
            [
                "open",
                "close",
                "high"
            ]
        ].min(axis=1)

    ).any():

        raise ValueError(
            "Lowの整合性エラーがあります。"
        )


    idx = out.index


    grid_bad = (

        (idx.minute % 15 != 0)

        |

        (idx.second != 0)

        |

        (idx.microsecond != 0)
    )


    if np.asarray(
        grid_bad
    ).any():

        examples = list(

            idx[
                np.asarray(grid_bad)
            ][:5]
        )


        raise ValueError(

            "15分グリッドでない"
            f"timestampがあります: "
            f"{examples}"
        )


    return out



bars_raw = find_bars()


BARS = normalize_bars(
    bars_raw
)


print(
    f"[DATA] Normalized rows: "
    f"{len(BARS):,}"
)

print(
    f"[DATA] Period: "
    f"{BARS.index.min()} "
    f"-> "
    f"{BARS.index.max()}"
)



# ============================================================
# 2. 特徴量作成
# ============================================================

def calc_rsi(
    close,
    period=14
):

    delta = close.diff()


    gain = delta.clip(
        lower=0
    )


    loss = -delta.clip(
        upper=0
    )


    avg_gain = gain.rolling(
        period
    ).mean()


    avg_loss = loss.rolling(
        period
    ).mean()


    rs = (

        avg_gain

        /

        avg_loss.replace(
            0,
            np.nan
        )
    )


    return (
        100
        -
        100 / (1 + rs)
    )



def true_range_series(
    x
):

    prev_close = x[
        "close"
    ].shift(1)


    return pd.concat(

        [

            x["high"]
            -
            x["low"],


            (
                x["high"]
                -
                prev_close
            ).abs(),


            (
                x["low"]
                -
                prev_close
            ).abs(),
        ],

        axis=1

    ).max(axis=1)



def calc_adx(
    x,
    period=14
):

    high = x["high"]

    low = x["low"]


    up_move = high.diff()

    down_move = -low.diff()


    plus_dm = pd.Series(

        np.where(

            (
                up_move
                >
                down_move
            )

            &

            (
                up_move
                >
                0
            ),

            up_move,

            0.0
        ),

        index=x.index
    )


    minus_dm = pd.Series(

        np.where(

            (
                down_move
                >
                up_move
            )

            &

            (
                down_move
                >
                0
            ),

            down_move,

            0.0
        ),

        index=x.index
    )


    tr = true_range_series(
        x
    )


    tr_sum = (

        tr.rolling(
            period
        )

        .sum()

        .replace(
            0,
            np.nan
        )
    )


    plus_di = (

        100

        *

        plus_dm.rolling(
            period
        ).sum()

        /

        tr_sum
    )


    minus_di = (

        100

        *

        minus_dm.rolling(
            period
        ).sum()

        /

        tr_sum
    )


    denom = (

        plus_di
        +
        minus_di

    ).replace(
        0,
        np.nan
    )


    dx = (

        100

        *

        (
            plus_di
            -
            minus_di
        ).abs()

        /

        denom
    )


    return (

        dx.rolling(
            period
        ).mean()

        /

        100.0
    )



def rolling_z(
    s,
    window
):

    mean = s.rolling(
        window
    ).mean()


    std = (

        s.rolling(
            window
        )

        .std()

        .replace(
            0,
            np.nan
        )
    )


    return (
        s - mean
    ) / std



def make_all_features(
    bars
):

    x = bars.copy()


    # --------------------------------------------------------
    # 正式Champion 30特徴量
    # --------------------------------------------------------

    for n in [
        1,
        2,
        4,
        8,
        16
    ]:

        x[
            f"return_{n}"
        ] = (

            x["close"]
            .pct_change(n)
        )


    for n in [
        4,
        8,
        16,
        32
    ]:

        x[
            f"vol_{n}"
        ] = (

            x["return_1"]

            .rolling(n)

            .std()
        )


    ma_cache = {}


    for period in [
        5,
        10,
        20,
        50,
        100
    ]:

        ma = (

            x["close"]

            .rolling(
                period
            )

            .mean()
        )


        ma_cache[
            period
        ] = ma


        x[
            f"ma{period}_distance"
        ] = (

            x["close"]

            /

            ma

            -

            1
        )


        x[
            f"ma{period}_slope"
        ] = ma.pct_change()


    candle_range = (

        x["high"]
        -
        x["low"]

    ).replace(
        0,
        np.nan
    )


    x["body"] = (

        x["close"]
        -
        x["open"]

    ) / candle_range


    x["upper_wick"] = (

        x["high"]

        -

        x[
            [
                "open",
                "close"
            ]
        ].max(
            axis=1
        )

    ) / candle_range


    x["lower_wick"] = (

        x[
            [
                "open",
                "close"
            ]
        ].min(
            axis=1
        )

        -

        x["low"]

    ) / candle_range


    x["range_pct"] = (

        x["high"]
        -
        x["low"]

    ) / x["close"]


    x["rsi14"] = (

        calc_rsi(
            x["close"],
            14
        )

        /

        100.0
    )


    tr = true_range_series(
        x
    )


    atr14_abs = (

        tr.rolling(
            14
        ).mean()
    )


    x["atr14"] = (

        atr14_abs

        /

        x["close"]
    )


    high16 = (

        x["high"]

        .rolling(
            16
        )

        .max()
    )


    low16 = (

        x["low"]

        .rolling(
            16
        )

        .min()
    )


    x[
        "distance_high_16"
    ] = (

        high16
        -
        x["close"]

    ) / x["close"]


    x[
        "distance_low_16"
    ] = (

        x["close"]
        -
        low16

    ) / x["close"]


    hour = (

        x.index.hour

        +

        x.index.minute
        /
        60.0
    )


    x["hour_sin"] = np.sin(

        2
        *
        np.pi
        *
        hour
        /
        24.0
    )


    x["hour_cos"] = np.cos(

        2
        *
        np.pi
        *
        hour
        /
        24.0
    )


    x["weekday"] = (

        x.index.dayofweek

        /

        4.0
    )


    # --------------------------------------------------------
    # Volatility 13
    # --------------------------------------------------------

    x["vol_64"] = (

        x["return_1"]

        .rolling(
            64
        )

        .std()
    )


    x["vol_96"] = (

        x["return_1"]

        .rolling(
            96
        )

        .std()
    )


    eps = 1e-12


    x[
        "vol_ratio_4_32"
    ] = (

        x["vol_4"]

        /

        (
            x["vol_32"].abs()
            +
            eps
        )
    )


    x[
        "vol_ratio_8_32"
    ] = (

        x["vol_8"]

        /

        (
            x["vol_32"].abs()
            +
            eps
        )
    )


    x[
        "vol_ratio_16_64"
    ] = (

        x["vol_16"]

        /

        (
            x["vol_64"].abs()
            +
            eps
        )
    )


    x[
        "range_mean_8"
    ] = (

        x["range_pct"]

        .rolling(
            8
        )

        .mean()
    )


    x[
        "range_mean_32"
    ] = (

        x["range_pct"]

        .rolling(
            32
        )

        .mean()
    )


    x[
        "range_ratio_8_32"
    ] = (

        x["range_mean_8"]

        /

        (
            x["range_mean_32"].abs()
            +
            eps
        )
    )


    x[
        "range_z_20"
    ] = rolling_z(

        x["range_pct"],
        20
    )


    x[
        "range_z_50"
    ] = rolling_z(

        x["range_pct"],
        50
    )


    abs_ret = x[
        "return_1"
    ].abs()


    x[
        "abs_return_z_32"
    ] = rolling_z(

        abs_ret,
        32
    )


    x[
        "abs_return_z_96"
    ] = rolling_z(

        abs_ret,
        96
    )


    x[
        "gap_abs_1"
    ] = (

        x["open"]

        /

        x["close"].shift(
            1
        )

        -

        1
    ).abs()


    # --------------------------------------------------------
    # Regime 11
    # --------------------------------------------------------

    x[
        "adx14"
    ] = calc_adx(
        x,
        14
    )


    x[
        "adx28"
    ] = calc_adx(
        x,
        28
    )


    ma20 = ma_cache[
        20
    ]


    ma50 = ma_cache[
        50
    ]


    ma100 = ma_cache[
        100
    ]


    x[
        "ma20_50_spread"
    ] = (

        ma20

        /

        ma50

        -

        1
    )


    x[
        "ma50_100_spread"
    ] = (

        ma50

        /

        ma100

        -

        1
    )


    atr_safe = (

        atr14_abs

        .replace(
            0,
            np.nan
        )
    )


    x[
        "trend_strength_20"
    ] = (

        (
            x["close"]
            -
            ma20
        ).abs()

        /

        atr_safe
    )


    x[
        "trend_strength_50"
    ] = (

        (
            x["close"]
            -
            ma50
        ).abs()

        /

        atr_safe
    )


    high20 = (

        x["high"]

        .rolling(
            20
        )

        .max()
    )


    low20 = (

        x["low"]

        .rolling(
            20
        )

        .min()
    )


    high50 = (

        x["high"]

        .rolling(
            50
        )

        .max()
    )


    low50 = (

        x["low"]

        .rolling(
            50
        )

        .min()
    )


    x[
        "price_pos_20"
    ] = (

        (
            x["close"]
            -
            low20
        )

        /

        (
            high20
            -
            low20

        ).replace(
            0,
            np.nan
        )

        -

        0.5
    )


    x[
        "price_pos_50"
    ] = (

        (
            x["close"]
            -
            low50
        )

        /

        (
            high50
            -
            low50

        ).replace(
            0,
            np.nan
        )

        -

        0.5
    )


    x[
        "ma_alignment_score"
    ] = (

        (
            ma20
            >
            ma50
        ).astype(float)

        +

        (
            ma50
            >
            ma100
        ).astype(float)

    ) / 2.0


    slope20 = ma20.pct_change()

    slope50 = ma50.pct_change()

    slope100 = ma100.pct_change()


    x[
        "slope_alignment_score"
    ] = (

        np.sign(
            slope20
        )

        +

        np.sign(
            slope50
        )

        +

        np.sign(
            slope100
        )

    ) / 3.0


    sign_ret = np.sign(
        x["return_1"]
    )


    x[
        "directional_persistence_16"
    ] = (

        sign_ret

        .rolling(
            16
        )

        .mean()

        .abs()
    )


    return x.replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )



# ============================================================
# 3. Target / 30分固定Exit
# ============================================================

def prepare_dataset(
    bars
):

    x = make_all_features(
        bars
    )


    times = pd.Series(

        bars.index,

        index=bars.index
    )


    # signal t
    # ↓
    # Open(t+1) entry
    # ↓
    # Close(t+2) exit
    # = 30分保有

    x[
        "entry_time"
    ] = times.shift(
        -1
    )


    x[
        "label_end"
    ] = (

        times.shift(
            -2
        )

        +

        pd.Timedelta(
            minutes=15
        )
    )


    x[
        "entry_price"
    ] = bars[
        "open"
    ].shift(
        -1
    )


    x[
        "exit_price"
    ] = bars[
        "close"
    ].shift(
        -2
    )


    x[
        "future_return"
    ] = (

        x["exit_price"]

        /

        x["entry_price"]

        -

        1
    )


    x[
        "target"
    ] = (

        x[
            "future_return"
        ]

        >

        0

    ).astype(
        int
    )


    # t,t+1,t+2 が
    # 本当に連続15分足か確認

    continuous = (

        (
            times.shift(-1)
            -
            times
        ).eq(
            pd.Timedelta(
                minutes=15
            )
        )

        &

        (
            times.shift(-2)
            -
            times
        ).eq(
            pd.Timedelta(
                minutes=30
            )
        )
    )


    # 3者を完全に同一行で比較するため
    # 全特徴量が揃った行だけ残す

    all_features = sorted(

        set(

            CHAMPION_FEATURES

            +

            VOLATILITY_FEATURES

            +

            REGIME_FEATURES
        )
    )


    required = (

        all_features

        +

        [
            "entry_time",
            "label_end",
            "entry_price",
            "exit_price",
            "future_return",
            "target",
        ]
    )


    out = (

        x.loc[
            continuous
        ]

        .dropna(
            subset=required
        )

        .copy()
    )


    if len(out) < 10000:

        raise RuntimeError(

            "使用可能行が少なすぎます: "
            f"{len(out):,}"
        )


    return out



DATA = prepare_dataset(
    BARS
)


print(
    f"[DATASET] Usable rows: "
    f"{len(DATA):,}"
)


print(
    "[DATASET] Years:",
    sorted(
        DATA.index.year.unique()
    )
)


print(
    "[DATASET] Target UP rate:",
    DATA[
        "target"
    ].mean()
)



# ============================================================
# 4. 正式HGB
# ============================================================

def fit_hgb(
    train,
    features
):

    if len(train) < MIN_TRAIN_ROWS:

        raise RuntimeError(

            "Training rows insufficient: "
            f"{len(train):,}"
        )


    if train[
        "target"
    ].nunique() < 2:

        raise RuntimeError(
            "Training target has only one class."
        )


    model = HistGradientBoostingClassifier(
        **HGB_CONFIG
    )


    model.fit(

        train[
            features
        ],

        train[
            "target"
        ]
    )


    return model



def raw_prob(
    model,
    frame,
    features
):

    classes = list(
        model.classes_
    )


    if 1 not in classes:

        raise RuntimeError(
            "HGB positive class 1 not found."
        )


    return model.predict_proba(

        frame[
            features
        ]

    )[
        :,
        classes.index(1)
    ]



# ============================================================
# 5. Raw expanding Walk-Forwardを事前計算
# ============================================================

def build_raw_walkforward(
    data,
    features,
    feature_set_name
):

    years = sorted(
        data.index.year.unique()
    )


    pieces = []


    print(
        "\n"
        +
        "=" * 100
    )


    print(
        "PRECOMPUTE RAW WALK-FORWARD:",
        feature_set_name
    )


    print(
        "=" * 100
    )


    for year in years:

        prior_years = [

            y

            for y in years

            if y < year
        ]


        if (
            len(prior_years)
            <
            MIN_PRIOR_YEARS_FOR_OOF
        ):

            continue


        start = pd.Timestamp(

            f"{year}-01-01",

            tz="UTC"
        )


        end = pd.Timestamp(

            f"{year+1}-01-01",

            tz="UTC"
        )


        # future labelが
        # 次年に跨がないようpurge

        hist = data.loc[

            (
                data.index
                <
                start
            )

            &

            (
                data[
                    "label_end"
                ]
                <=
                start
            )

        ].copy()


        test = data.loc[

            (
                data.index
                >=
                start
            )

            &

            (
                data.index
                <
                end
            )

            &

            (
                data[
                    "label_end"
                ]
                <=
                end
            )

        ].copy()


        if (
            len(hist)
            <
            MIN_TRAIN_ROWS
        ):

            continue


        if (
            len(test)
            <
            MIN_EVAL_ROWS
        ):

            continue


        model = fit_hgb(

            hist,

            features
        )


        p = raw_prob(

            model,

            test,

            features
        )


        piece = pd.DataFrame(

            {

                "raw_probability":
                    p,

                "target":
                    test[
                        "target"
                    ].to_numpy(
                        dtype=int
                    ),
            },

            index=test.index
        )


        piece[
            "year"
        ] = year


        pieces.append(
            piece
        )


        if year >= 2018:

            auc = (

                roc_auc_score(

                    test[
                        "target"
                    ],

                    p
                )

                if

                test[
                    "target"
                ].nunique()
                ==
                2

                else

                np.nan
            )


            print(

                f"{year}: "
                f"rows={len(test):,}, "
                f"raw AUC={auc:.4f}"
            )


    if not pieces:

        raise RuntimeError(

            f"{feature_set_name}: "
            "Walk-forward予測を"
            "作成できませんでした。"
        )


    return pd.concat(
        pieces
    ).sort_index()



RAW_WF = {}


for (
    fs_name,
    fs_features
) in FEATURE_SETS.items():

    RAW_WF[
        fs_name
    ] = build_raw_walkforward(

        DATA,

        fs_features,

        fs_name
    )



# ============================================================
# 6. Calibration
# ============================================================

class IdentityCalibrator:

    def fit(
        self,
        p,
        y
    ):

        return self


    def predict(
        self,
        p
    ):

        return np.asarray(
            p,
            dtype=float
        )



class PlattCalibrator:

    def __init__(
        self
    ):

        self.model = LogisticRegression(

            C=1.0,

            solver="lbfgs",

            random_state=RANDOM_SEED
        )


    def fit(
        self,
        p,
        y
    ):

        self.model.fit(

            np.asarray(
                p
            ).reshape(
                -1,
                1
            ),

            np.asarray(
                y
            )
        )


        return self


    def predict(
        self,
        p
    ):

        return self.model.predict_proba(

            np.asarray(
                p
            ).reshape(
                -1,
                1
            )

        )[
            :,
            1
        ]



class IsotonicCalibrator:

    def __init__(
        self
    ):

        self.model = IsotonicRegression(

            y_min=0.0,

            y_max=1.0,

            out_of_bounds="clip"
        )


    def fit(
        self,
        p,
        y
    ):

        self.model.fit(

            np.asarray(
                p
            ),

            np.asarray(
                y
            )
        )


        return self


    def predict(
        self,
        p
    ):

        return np.asarray(

            self.model.predict(

                np.asarray(
                    p
                )
            ),

            dtype=float
        )



def fit_calibrator(
    method,
    oof
):

    if method == "RAW":

        return IdentityCalibrator()


    if (

        len(oof)
        <
        500

        or

        oof[
            "target"
        ].nunique()
        <
        2

    ):

        return IdentityCalibrator()


    if method == "PLATT":

        return PlattCalibrator().fit(

            oof[
                "raw_probability"
            ].to_numpy(),

            oof[
                "target"
            ].to_numpy()
        )


    if method == "ISOTONIC":

        if len(oof) < 1000:

            return IdentityCalibrator()


        return IsotonicCalibrator().fit(

            oof[
                "raw_probability"
            ].to_numpy(),

            oof[
                "target"
            ].to_numpy()
        )


    raise ValueError(
        method
    )



def expected_calibration_error(
    y_true,
    p,
    bins=10
):

    y = np.asarray(
        y_true,
        dtype=float
    )


    p = np.clip(

        np.asarray(
            p,
            dtype=float
        ),

        0,

        1
    )


    if len(y) == 0:

        return np.nan


    edges = np.linspace(

        0,
        1,
        bins + 1
    )


    ids = np.clip(

        np.digitize(

            p,

            edges[
                1:-1
            ],

            right=False
        ),

        0,

        bins - 1
    )


    ece = 0.0


    for b in range(
        bins
    ):

        mask = (
            ids == b
        )


        if mask.any():

            ece += (

                mask.mean()

                *

                abs(

                    y[
                        mask
                    ].mean()

                    -

                    p[
                        mask
                    ].mean()
                )
            )


    return float(
        ece
    )



def calibration_metrics(
    y_true,
    p
):

    y = np.asarray(
        y_true,
        dtype=int
    )


    p = np.clip(

        np.asarray(
            p,
            dtype=float
        ),

        1e-8,

        1 - 1e-8
    )


    auc = (

        roc_auc_score(
            y,
            p
        )

        if

        len(
            np.unique(y)
        )
        ==
        2

        else

        np.nan
    )


    return {

        "brier":
            float(
                brier_score_loss(
                    y,
                    p
                )
            ),

        "ece":
            expected_calibration_error(
                y,
                p,
                bins=10
            ),

        "auc":
            float(
                auc
            )
    }



def raw_year(
    raw_wf,
    year
):

    return raw_wf.loc[

        raw_wf[
            "year"
        ]
        ==
        year

    ].copy()



def raw_oof_before(
    raw_wf,
    cutoff_year
):

    return raw_wf.loc[

        raw_wf[
            "year"
        ]
        <
        cutoff_year,

        [
            "raw_probability",
            "target"
        ]

    ].copy()



def choose_calibration(
    raw_wf,
    validation_year
):

    oof = raw_oof_before(

        raw_wf,

        validation_year
    )


    val = raw_year(

        raw_wf,

        validation_year
    )


    if len(val) < MIN_EVAL_ROWS:

        raise RuntimeError(

            "Validation予測が不足: "
            f"{validation_year}"
        )


    raw_val = val[
        "raw_probability"
    ].to_numpy()


    y_val = val[
        "target"
    ].to_numpy()


    rows = []


    for method in CALIBRATION_METHODS:

        cal = fit_calibrator(

            method,

            oof
        )


        p = np.clip(

            cal.predict(
                raw_val
            ),

            0,

            1
        )


        m = calibration_metrics(

            y_val,

            p
        )


        rows.append(

            {
                "method":
                    method,

                **m
            }
        )


    table = (

        pd.DataFrame(
            rows
        )

        .sort_values(

            [
                "brier",
                "ece",
                "method"
            ]
        )
    )


    chosen = str(

        table.iloc[
            0
        ][
            "method"
        ]
    )


    return (
        chosen,
        table
    )



# ============================================================
# 7. Trade selection
# ============================================================

def session_mask(
    index,
    policy
):

    hour = pd.DatetimeIndex(
        index
    ).hour


    if policy == "ALL":

        return np.ones(
            len(index),
            dtype=bool
        )


    if policy == "UTC_13_24":

        return (

            (hour >= 13)

            &

            (hour < 24)
        )


    if policy == "UTC_21_24":

        return (

            (hour >= 21)

            &

            (hour < 24)
        )


    if policy == "EXCLUDE_08_13":

        return ~(

            (hour >= 8)

            &

            (hour < 13)
        )


    raise ValueError(
        policy
    )



def predictions_frame(
    frame,
    calibrated_p_up,
    raw_p_up
):

    p = np.asarray(

        calibrated_p_up,

        dtype=float
    )


    raw_p = np.asarray(

        raw_p_up,

        dtype=float
    )


    direction_sign = np.where(

        p >= 0.5,

        1.0,

        -1.0
    )


    out = frame[
        [
            "entry_time",
            "label_end",
            "entry_price",
            "exit_price",
            "future_return",
            "target",
        ]
    ].copy()


    out[
        "p_up"
    ] = p


    out[
        "raw_p_up"
    ] = raw_p


    out[
        "confidence"
    ] = np.maximum(

        p,

        1 - p
    )


    out[
        "direction"
    ] = np.where(

        p >= 0.5,

        "BUY",

        "SELL"
    )


    out[
        "direction_correct"
    ] = (

        (
            p >= 0.5
        )

        ==

        (
            frame[
                "future_return"
            ].to_numpy()

            >

            0
        )
    )


    out[
        "gross_return"
    ] = (

        frame[
            "future_return"
        ].to_numpy(
            dtype=float
        )

        *

        direction_sign
    )


    return out



def select_trades(
    predictions,
    threshold,
    session_policy
):

    mask = (

        (
            predictions[
                "confidence"
            ]
            >=
            threshold
        )

        &

        session_mask(

            predictions.index,

            session_policy
        )
    )


    candidates = (

        predictions.loc[
            mask
        ]

        .sort_index()
    )


    selected = []


    next_free_time = None


    for row in candidates.itertuples():

        # 同時ポジション禁止

        if (

            next_free_time
            is not None

            and

            row.entry_time
            <
            next_free_time
        ):

            continue


        selected.append(
            row.Index
        )


        next_free_time = (
            row.label_end
        )


    trades = candidates.loc[
        selected
    ].copy()


    trades.index.name = (
        "signal_time"
    )


    trades[
        "base_net_return"
    ] = (

        trades[
            "gross_return"
        ]

        -

        COST
    )


    return trades



# ============================================================
# 8. Metrics
# ============================================================

def basic_stats(
    returns
):

    r = np.asarray(

        returns,

        dtype=float
    )


    r = r[
        np.isfinite(r)
    ]


    if len(r) == 0:

        return {

            "trades":
                0,

            "win_rate":
                np.nan,

            "avg_return":
                np.nan,

            "median_return":
                np.nan,

            "profit_factor":
                np.nan,

            "growth":
                0.0,

            "max_dd":
                np.nan,

            "return_to_dd":
                np.nan,
        }


    gains = r[
        r > 0
    ].sum()


    losses = -r[
        r < 0
    ].sum()


    if losses > 0:

        pf = gains / losses


    elif gains > 0:

        pf = np.inf


    else:

        pf = np.nan


    equity = np.r_[

        1.0,

        np.cumprod(
            1 + r
        )
    ]


    peak = np.maximum.accumulate(
        equity
    )


    dd = (

        equity
        /
        peak
        -
        1
    )


    growth = float(

        equity[
            -1
        ]

        -

        1
    )


    max_dd = float(
        dd.min()
    )


    rdd = (

        growth
        /
        abs(
            max_dd
        )

        if

        np.isfinite(
            max_dd
        )

        and

        max_dd < 0

        else

        np.nan
    )


    return {

        "trades":
            int(
                len(r)
            ),

        "win_rate":
            float(
                (
                    r > 0
                ).mean()
            ),

        "avg_return":
            float(
                r.mean()
            ),

        "median_return":
            float(
                np.median(r)
            ),

        "profit_factor":
            float(
                pf
            ),

        "growth":
            growth,

        "max_dd":
            max_dd,

        "return_to_dd":
            float(
                rdd
            )
    }



# ============================================================
# 9. Threshold + Session
# ============================================================

def choose_threshold_and_session(
    validation_predictions
):

    rows = []


    best = None

    best_key = None


    for threshold in THRESHOLDS:

        for session in SESSION_POLICIES:

            trades = select_trades(

                validation_predictions,

                threshold,

                session
            )


            stats = basic_stats(

                trades[
                    "base_net_return"
                ]
            )


            eligible = (

                stats[
                    "trades"
                ]

                >=

                MIN_VALIDATION_TRADES
            )


            score = (

                stats[
                    "avg_return"
                ]

                *

                math.sqrt(

                    stats[
                        "trades"
                    ]
                )

                if

                eligible

                and

                np.isfinite(

                    stats[
                        "avg_return"
                    ]
                )

                else

                np.nan
            )


            rows.append(

                {

                    "threshold":
                        threshold,

                    "session":
                        session,

                    "eligible":
                        eligible,

                    "score":
                        score,

                    **stats
                }
            )


            if not eligible:

                continue


            key = (

                float(
                    score
                ),

                float(

                    stats[
                        "profit_factor"
                    ]

                )

                if

                np.isfinite(

                    stats[
                        "profit_factor"
                    ]
                )

                else

                -np.inf,

                int(

                    stats[
                        "trades"
                    ]
                ),

                -float(
                    threshold
                )
            )


            if (

                best_key is None

                or

                key > best_key
            ):

                best_key = key


                best = (

                    float(
                        threshold
                    ),

                    str(
                        session
                    )
                )


    table = pd.DataFrame(
        rows
    )


    if best is None:

        return (
            None,
            None,
            table
        )


    return (

        best[
            0
        ],

        best[
            1
        ],

        table
    )



# ============================================================
# 10. Adaptive Position Sizing
# ============================================================

def raw_position_size(
    confidence,
    threshold,
    policy
):

    c = np.asarray(

        confidence,

        dtype=float
    )


    denom = max(

        1.0
        -
        threshold,

        1e-8
    )


    edge = np.clip(

        (
            c
            -
            threshold
        )

        /

        denom,

        0,

        1
    )


    if policy == "FIXED":

        return np.ones_like(
            edge
        )


    if policy == "GENTLE":

        return (

            0.85

            +

            0.30
            *
            edge
        )


    if policy == "MODERATE":

        return (

            0.70

            +

            0.60
            *
            edge
        )


    if policy == "STRONG":

        return (

            0.50

            +

            1.00
            *
            edge
        )


    raise ValueError(
        policy
    )



def sizing_candidate_stats(
    trades,
    threshold,
    policy
):

    if trades.empty:

        return None


    raw = raw_position_size(

        trades[
            "confidence"
        ].to_numpy(),

        threshold,

        policy
    )


    if (

        not np.isfinite(
            raw
        ).all()

        or

        raw.mean() <= 0
    ):

        return None


    scale = (

        1.0

        /

        raw.mean()
    )


    size = (

        raw

        *

        scale
    )


    sized_return = (

        size

        *

        (

            trades[
                "gross_return"
            ].to_numpy(
                dtype=float
            )

            -

            COST
        )
    )


    stats = basic_stats(
        sized_return
    )


    return {

        "policy":
            policy,

        "validation_scale":
            float(
                scale
            ),

        "mean_size":
            float(
                size.mean()
            ),

        **stats
    }



def choose_sizing(
    validation_trades,
    threshold
):

    rows = []


    for policy in SIZING_POLICIES:

        row = sizing_candidate_stats(

            validation_trades,

            threshold,

            policy
        )


        if row is not None:

            rows.append(
                row
            )


    table = pd.DataFrame(
        rows
    )


    if (

        table.empty

        or

        "FIXED"
        not in
        set(
            table[
                "policy"
            ]
        )

    ):

        return (
            "FIXED",
            1.0,
            table
        )


    fixed = table.loc[

        table[
            "policy"
        ]

        ==

        "FIXED"

    ].iloc[
        0
    ]


    candidates = []


    for _, row in table.iterrows():

        if row[
            "policy"
        ] == "FIXED":

            continue


        avg_ok = (

            row[
                "avg_return"
            ]

            >=

            fixed[
                "avg_return"
            ]
        )


        pf_ok = (

            row[
                "profit_factor"
            ]

            >=

            fixed[
                "profit_factor"
            ]
        )


        if (

            np.isfinite(

                row[
                    "return_to_dd"
                ]
            )

            and

            np.isfinite(

                fixed[
                    "return_to_dd"
                ]
            )

        ):

            rdd_ok = (

                row[
                    "return_to_dd"
                ]

                >=

                fixed[
                    "return_to_dd"
                ]
            )


        else:

            rdd_ok = False


        if (

            avg_ok

            and

            pf_ok

            and

            rdd_ok
        ):

            candidates.append(
                row
            )


    if not candidates:

        chosen = fixed


    else:

        chosen = max(

            candidates,

            key=lambda r: (

                r[
                    "return_to_dd"
                ],

                r[
                    "profit_factor"
                ],

                r[
                    "avg_return"
                ]
            )
        )


    return (

        str(
            chosen[
                "policy"
            ]
        ),

        float(
            chosen[
                "validation_scale"
            ]
        ),

        table
    )



def apply_sizing(
    trades,
    threshold,
    policy,
    validation_scale,
    cost_multiplier=1.0
):

    out = trades.copy()


    raw = raw_position_size(

        out[
            "confidence"
        ].to_numpy(),

        threshold,

        policy
    )


    size = (

        raw

        *

        validation_scale
    )


    size = np.clip(

        size,

        0.25,

        2.0
    )


    out[
        "position_size"
    ] = size


    out[
        "cost_multiplier"
    ] = float(
        cost_multiplier
    )


    out[
        "net_return_1x"
    ] = (

        out[
            "gross_return"
        ]

        -

        COST
        *
        cost_multiplier
    )


    out[
        "sized_return"
    ] = (

        size

        *

        out[
            "net_return_1x"
        ]
    )


    return out



# ============================================================
# 11. Nested年次評価
# ============================================================

def evaluate_one_year(
    feature_set_name,
    test_year
):

    raw_wf = RAW_WF[
        feature_set_name
    ]


    validation_year = (

        test_year

        -

        1
    )


    raw_val_df = raw_year(

        raw_wf,

        validation_year
    )


    raw_test_df = raw_year(

        raw_wf,

        test_year
    )


    if (

        len(raw_val_df)
        <
        MIN_EVAL_ROWS

        or

        len(raw_test_df)
        <
        MIN_EVAL_ROWS
    ):

        return (
            None,
            pd.DataFrame()
        )


    validation = DATA.loc[

        raw_val_df.index

    ].copy()


    test = DATA.loc[

        raw_test_df.index

    ].copy()


    # --------------------------------------------------------
    # A. Calibration
    # --------------------------------------------------------

    calibration, cal_table = choose_calibration(

        raw_wf,

        validation_year
    )


    oof_train = raw_oof_before(

        raw_wf,

        validation_year
    )


    val_calibrator = fit_calibrator(

        calibration,

        oof_train
    )


    raw_val = raw_val_df[

        "raw_probability"

    ].to_numpy()


    p_val = np.clip(

        val_calibrator.predict(
            raw_val
        ),

        0,

        1
    )


    val_predictions = predictions_frame(

        validation,

        p_val,

        raw_val
    )


    # --------------------------------------------------------
    # B. Threshold + Session
    # --------------------------------------------------------

    threshold, session, threshold_table = (

        choose_threshold_and_session(

            val_predictions
        )
    )


    if threshold is None:

        print(

            f"[SKIP] "
            f"{feature_set_name} "
            f"{test_year}: "
            "Validationで最低取引数を"
            "満たす候補なし"
        )


        return (
            None,
            pd.DataFrame()
        )


    validation_selected = select_trades(

        val_predictions,

        threshold,

        session
    )


    # --------------------------------------------------------
    # C. Sizing
    # --------------------------------------------------------

    (

        sizing_policy,

        validation_scale,

        sizing_table

    ) = choose_sizing(

        validation_selected,

        threshold
    )


    # --------------------------------------------------------
    # D. Test前calibration再fit
    # --------------------------------------------------------

    final_oof = raw_oof_before(

        raw_wf,

        test_year
    )


    final_calibrator = fit_calibrator(

        calibration,

        final_oof
    )


    raw_test = raw_test_df[

        "raw_probability"

    ].to_numpy()


    p_test = np.clip(

        final_calibrator.predict(
            raw_test
        ),

        0,

        1
    )


    test_predictions = predictions_frame(

        test,

        p_test,

        raw_test
    )


    selected = select_trades(

        test_predictions,

        threshold,

        session
    )


    final_trades = apply_sizing(

        selected,

        threshold,

        sizing_policy,

        validation_scale,

        cost_multiplier=1.0
    )


    stats = basic_stats(

        final_trades[
            "sized_return"
        ]
    )


    cal_metrics = calibration_metrics(

        test[
            "target"
        ].to_numpy(),

        p_test
    )


    raw_auc = (

        roc_auc_score(

            test[
                "target"
            ],

            raw_test
        )

        if

        test[
            "target"
        ].nunique()
        ==
        2

        else

        np.nan
    )


    result = {

        "feature_set":
            feature_set_name,

        "test_year":
            int(
                test_year
            ),

        "validation_year":
            int(
                validation_year
            ),

        "calibration":
            calibration,

        "threshold":
            float(
                threshold
            ),

        "session":
            session,

        "sizing_policy":
            sizing_policy,

        "validation_scale":
            float(
                validation_scale
            ),

        "raw_auc":
            float(
                raw_auc
            ),

        "calibrated_auc":
            float(
                cal_metrics[
                    "auc"
                ]
            ),

        "test_brier":
            float(
                cal_metrics[
                    "brier"
                ]
            ),

        "test_ece":
            float(
                cal_metrics[
                    "ece"
                ]
            ),

        "mean_position_size":
            (

                float(

                    final_trades[
                        "position_size"
                    ].mean()
                )

                if

                len(
                    final_trades
                )

                else

                np.nan
            ),

        **stats
    }


    final_trades = (
        final_trades.copy()
    )


    final_trades[
        "feature_set"
    ] = feature_set_name


    final_trades[
        "test_year"
    ] = int(
        test_year
    )


    final_trades[
        "calibration"
    ] = calibration


    final_trades[
        "threshold"
    ] = float(
        threshold
    )


    final_trades[
        "session"
    ] = session


    final_trades[
        "sizing_policy"
    ] = sizing_policy


    return (
        result,
        final_trades
    )



# ============================================================
# 12. 3者対決
# ============================================================

annual_rows = []

trade_frames = []


for fs_name in FEATURE_SETS:

    print(
        "\n"
        +
        "#" * 100
    )


    print(
        "RUNNING FEATURE SET:",
        fs_name
    )


    print(
        "#" * 100
    )


    for year in (

        list(
            DEVELOPMENT_YEARS
        )

        +

        [
            CONFIRMATION_YEAR
        ]
    ):

        result, trades = evaluate_one_year(

            fs_name,

            year
        )


        if result is None:

            continue


        annual_rows.append(
            result
        )


        if not trades.empty:

            trade_frames.append(
                trades
            )


        print(

            f"{year} | "

            f"Cal={result['calibration']} "

            f"Thr={result['threshold']:.2f} "

            f"Session={result['session']} "

            f"Size={result['sizing_policy']} | "

            f"AUC={result['calibrated_auc']:.4f} "

            f"Trades={result['trades']} "

            f"PF={result['profit_factor']:.3f} "

            f"Avg={result['avg_return']*100:.5f}% "

            f"Growth={result['growth']*100:.3f}% "

            f"DD={result['max_dd']*100:.3f}%"
        )



ANNUAL = pd.DataFrame(
    annual_rows
)


TRADES = (

    pd.concat(
        trade_frames
    ).sort_index()

    if

    trade_frames

    else

    pd.DataFrame()
)


if ANNUAL.empty:

    raise RuntimeError(

        "Annual evaluation produced "
        "no results."
    )



# ============================================================
# 13. Development Summary / 2026
# ============================================================

def aggregate_feature_set(
    feature_set,
    years
):

    a = ANNUAL.loc[

        (
            ANNUAL[
                "feature_set"
            ]
            ==
            feature_set
        )

        &

        (
            ANNUAL[
                "test_year"
            ].isin(
                years
            )
        )

    ].copy()


    if TRADES.empty:

        return None


    t = TRADES.loc[

        (
            TRADES[
                "feature_set"
            ]
            ==
            feature_set
        )

        &

        (
            TRADES[
                "test_year"
            ].isin(
                years
            )
        )

    ].copy()


    if a.empty or t.empty:

        return None


    stats = basic_stats(

        t[
            "sized_return"
        ]
    )


    return {

        "feature_set":
            feature_set,

        "years":
            int(
                a[
                    "test_year"
                ].nunique()
            ),

        "mean_auc":
            float(
                a[
                    "calibrated_auc"
                ].mean()
            ),

        "median_auc":
            float(
                a[
                    "calibrated_auc"
                ].median()
            ),

        "positive_years":
            int(

                (
                    a[
                        "avg_return"
                    ]
                    >
                    0
                ).sum()
            ),

        "pf_above_1_years":
            int(

                (
                    a[
                        "profit_factor"
                    ]
                    >
                    1
                ).sum()
            ),

        **stats
    }



dev_rows = []


for fs_name in FEATURE_SETS:

    row = aggregate_feature_set(

        fs_name,

        DEVELOPMENT_YEARS
    )


    if row is not None:

        dev_rows.append(
            row
        )


DEV_SUMMARY = pd.DataFrame(
    dev_rows
)


CONFIRMATION = ANNUAL.loc[

    ANNUAL[
        "test_year"
    ]
    ==
    CONFIRMATION_YEAR

].copy()



# ============================================================
# 14. Cost Stress
# ============================================================

cost_rows = []


for fs_name in FEATURE_SETS:

    m = TRADES.loc[

        (
            TRADES[
                "feature_set"
            ]
            ==
            fs_name
        )

        &

        (
            TRADES[
                "test_year"
            ].isin(
                DEVELOPMENT_YEARS
            )
        )

    ].copy()


    if m.empty:

        continue


    for mult in COST_MULTIPLIERS:

        r = (

            m[
                "position_size"
            ].to_numpy(
                dtype=float
            )

            *

            (

                m[
                    "gross_return"
                ].to_numpy(
                    dtype=float
                )

                -

                COST
                *
                mult
            )
        )


        stats = basic_stats(
            r
        )


        cost_rows.append(

            {

                "feature_set":
                    fs_name,

                "cost_x":
                    float(
                        mult
                    ),

                "cost_pct":
                    COST
                    *
                    mult,

                **stats
            }
        )


COST_STRESS = pd.DataFrame(
    cost_rows
)



# ============================================================
# 15. Paired Moving-Block Bootstrap
# ============================================================

def daily_return_series(
    trades
):

    if trades.empty:

        return pd.Series(
            dtype=float
        )


    x = trades.copy()


    x[
        "date"
    ] = (

        pd.DatetimeIndex(
            x.index
        )

        .normalize()
    )


    return (

        x.groupby(
            "date"
        )[
            "sized_return"
        ]

        .apply(

            lambda s:

            float(

                np.prod(

                    1

                    +

                    s.to_numpy(
                        dtype=float
                    )
                )

                -

                1
            )
        )
    )



def paired_block_bootstrap(
    candidate,
    iterations=BOOTSTRAP_ITERATIONS,
    block_days=BOOTSTRAP_BLOCK_DAYS
):

    base = TRADES.loc[

        (
            TRADES[
                "feature_set"
            ]
            ==
            "CHAMPION"
        )

        &

        (
            TRADES[
                "test_year"
            ].isin(
                DEVELOPMENT_YEARS
            )
        )

    ].copy()


    cand = TRADES.loc[

        (
            TRADES[
                "feature_set"
            ]
            ==
            candidate
        )

        &

        (
            TRADES[
                "test_year"
            ].isin(
                DEVELOPMENT_YEARS
            )
        )

    ].copy()


    if base.empty or cand.empty:

        return {

            "candidate":
                candidate,

            "observed_daily_alpha":
                np.nan,

            "ci_low":
                np.nan,

            "ci_high":
                np.nan,

            "prob_alpha_positive":
                np.nan,

            "days":
                0
        }


    b = daily_return_series(
        base
    )


    c = daily_return_series(
        cand
    )


    idx = (

        b.index

        .union(
            c.index
        )

        .sort_values()
    )


    delta = (

        c.reindex(
            idx,
            fill_value=0.0
        ).to_numpy()

        -

        b.reindex(
            idx,
            fill_value=0.0
        ).to_numpy()
    )


    n = len(
        delta
    )


    if n < max(
        30,
        block_days * 3
    ):

        return {

            "candidate":
                candidate,

            "observed_daily_alpha":
                float(
                    delta.mean()
                )
                if n
                else np.nan,

            "ci_low":
                np.nan,

            "ci_high":
                np.nan,

            "prob_alpha_positive":
                np.nan,

            "days":
                int(
                    n
                )
        }


    rng = np.random.default_rng(
        RANDOM_SEED
    )


    starts = np.arange(

        max(
            1,
            n - block_days + 1
        )
    )


    means = np.empty(

        iterations,

        dtype=float
    )


    for i in range(
        iterations
    ):

        sample = []


        while len(
            sample
        ) < n:

            s = int(

                rng.choice(
                    starts
                )
            )


            sample.extend(

                delta[
                    s:
                    s + block_days
                ]
            )


        means[
            i
        ] = (

            np.asarray(

                sample[
                    :n
                ],

                dtype=float
            )

            .mean()
        )


    low, high = np.percentile(

        means,

        [
            2.5,
            97.5
        ]
    )


    return {

        "candidate":
            candidate,

        "observed_daily_alpha":
            float(
                delta.mean()
            ),

        "ci_low":
            float(
                low
            ),

        "ci_high":
            float(
                high
            ),

        "prob_alpha_positive":
            float(
                (
                    means > 0
                ).mean()
            ),

        "days":
            int(
                n
            )
    }



BOOTSTRAP = pd.DataFrame(

    [

        paired_block_bootstrap(
            "CHAMPION_PLUS_REGIME"
        ),

        paired_block_bootstrap(
            "CHAMPION_PLUS_VOL_REGIME"
        ),
    ]
)



# ============================================================
# 16. 自動判定
# ============================================================

def get_dev_row(
    name
):

    q = DEV_SUMMARY.loc[

        DEV_SUMMARY[
            "feature_set"
        ]
        ==
        name
    ]


    return (

        q.iloc[
            0
        ]

        if len(q)

        else None
    )



def get_conf_row(
    name
):

    q = CONFIRMATION.loc[

        CONFIRMATION[
            "feature_set"
        ]
        ==
        name
    ]


    return (

        q.iloc[
            0
        ]

        if len(q)

        else None
    )



def get_cost_pf(
    name,
    mult=2.0
):

    q = COST_STRESS.loc[

        (
            COST_STRESS[
                "feature_set"
            ]
            ==
            name
        )

        &

        np.isclose(

            COST_STRESS[
                "cost_x"
            ],

            mult
        )

    ]


    return (

        float(

            q.iloc[
                0
            ][
                "profit_factor"
            ]
        )

        if len(q)

        else np.nan
    )



base_dev = get_dev_row(
    "CHAMPION"
)


base_conf = get_conf_row(
    "CHAMPION"
)


decision_rows = []


for candidate in [

    "CHAMPION_PLUS_REGIME",

    "CHAMPION_PLUS_VOL_REGIME"
]:

    d = get_dev_row(
        candidate
    )


    c = get_conf_row(
        candidate
    )


    if (

        d is None

        or

        c is None

        or

        base_dev is None

        or

        base_conf is None
    ):

        decision_rows.append(

            {

                "candidate":
                    candidate,

                "dev_wins":
                    0,

                "confirm_wins":
                    0,

                "cost_2x_pf":
                    np.nan,

                "bootstrap_p":
                    np.nan,

                "pass":
                    False
            }
        )


        continue


    dev_checks = {

        "AUC":

            d[
                "mean_auc"
            ]

            >

            base_dev[
                "mean_auc"
            ],


        "PF":

            d[
                "profit_factor"
            ]

            >

            base_dev[
                "profit_factor"
            ],


        "AVG_RETURN":

            d[
                "avg_return"
            ]

            >

            base_dev[
                "avg_return"
            ],


        "RETURN_DD":

            d[
                "return_to_dd"
            ]

            >

            base_dev[
                "return_to_dd"
            ],


        "POSITIVE_YEARS":

            d[
                "positive_years"
            ]

            >=

            base_dev[
                "positive_years"
            ]
    }


    conf_checks = {

        "AUC":

            c[
                "calibrated_auc"
            ]

            >

            base_conf[
                "calibrated_auc"
            ],


        "PF":

            c[
                "profit_factor"
            ]

            >

            base_conf[
                "profit_factor"
            ],


        "AVG_RETURN":

            c[
                "avg_return"
            ]

            >

            base_conf[
                "avg_return"
            ],


        "RETURN_DD":

            c[
                "return_to_dd"
            ]

            >

            base_conf[
                "return_to_dd"
            ]
    }


    dev_wins = int(

        sum(

            bool(v)

            for v in
            dev_checks.values()
        )
    )


    confirm_wins = int(

        sum(

            bool(v)

            for v in
            conf_checks.values()
        )
    )


    cost_2x_pf = get_cost_pf(

        candidate,

        2.0
    )


    b = BOOTSTRAP.loc[

        BOOTSTRAP[
            "candidate"
        ]
        ==
        candidate
    ]


    bootstrap_p = (

        float(

            b.iloc[
                0
            ][
                "prob_alpha_positive"
            ]
        )

        if len(b)

        else np.nan
    )


    passed = (

        dev_wins >= 3

        and

        confirm_wins >= 3

        and

        np.isfinite(
            cost_2x_pf
        )

        and

        cost_2x_pf > 1.0

        and

        np.isfinite(
            bootstrap_p
        )

        and

        bootstrap_p >= 0.80
    )


    decision_rows.append(

        {

            "candidate":
                candidate,

            "dev_wins":
                dev_wins,

            "confirm_wins":
                confirm_wins,

            "cost_2x_pf":
                cost_2x_pf,

            "bootstrap_p":
                bootstrap_p,

            "dev_checks":
                dev_checks,

            "confirm_checks":
                conf_checks,

            "pass":
                bool(
                    passed
                )
        }
    )



DECISION_TABLE = pd.DataFrame(
    decision_rows
)



passed = DECISION_TABLE.loc[

    DECISION_TABLE[
        "pass"
    ]
    ==
    True

].copy()



if passed.empty:

    FINAL_DECISION = (
        "KEEP_FORMAL_CHAMPION_30_FEATURES"
    )


else:

    passed = passed.sort_values(

        [

            "dev_wins",

            "confirm_wins",

            "cost_2x_pf",

            "bootstrap_p"
        ],

        ascending=False
    )


    FINAL_DECISION = (

        "PROMOTE_"

        +

        str(

            passed.iloc[
                0
            ][
                "candidate"
            ]
        )
    )



# ============================================================
# 17. 結果表示
# ============================================================

print(
    "\n"
    +
    "=" * 100
)


print(
    "ANNUAL RESULTS"
)


print(
    "=" * 100
)


show_cols = [

    "feature_set",

    "test_year",

    "calibration",

    "threshold",

    "session",

    "sizing_policy",

    "calibrated_auc",

    "trades",

    "win_rate",

    "avg_return",

    "profit_factor",

    "growth",

    "max_dd",

    "return_to_dd",
]


annual_show = ANNUAL[
    show_cols
].copy()


for c in [

    "win_rate",

    "avg_return",

    "growth",

    "max_dd"
]:

    annual_show[
        c
    ] = (

        annual_show[
            c
        ]

        *

        100
    )


print(
    annual_show.to_string(
        index=False
    )
)



print(
    "\n"
    +
    "=" * 100
)


print(
    "DEVELOPMENT SUMMARY 2020-2025"
)


print(
    "=" * 100
)


dev_show = DEV_SUMMARY.copy()


for c in [

    "win_rate",

    "avg_return",

    "median_return",

    "growth",

    "max_dd"
]:

    if c in dev_show:

        dev_show[
            c
        ] = (

            dev_show[
                c
            ]

            *

            100
        )


print(
    dev_show.to_string(
        index=False
    )
)



print(
    "\n"
    +
    "=" * 100
)


print(
    "CONFIRMATION 2026"
)


print(
    "=" * 100
)


conf_show = CONFIRMATION[
    show_cols
].copy()


for c in [

    "win_rate",

    "avg_return",

    "growth",

    "max_dd"
]:

    conf_show[
        c
    ] = (

        conf_show[
            c
        ]

        *

        100
    )


print(
    conf_show.to_string(
        index=False
    )
)



print(
    "\n"
    +
    "=" * 100
)


print(
    "COST STRESS: DEVELOPMENT OOS"
)


print(
    "=" * 100
)


cost_show = COST_STRESS.copy()


for c in [

    "cost_pct",

    "win_rate",

    "avg_return",

    "median_return",

    "growth",

    "max_dd"
]:

    if c in cost_show:

        cost_show[
            c
        ] = (

            cost_show[
                c
            ]

            *

            100
        )


print(
    cost_show.to_string(
        index=False
    )
)



print(
    "\n"
    +
    "=" * 100
)


print(
    "PAIRED BLOCK BOOTSTRAP vs FORMAL CHAMPION"
)


print(
    "=" * 100
)


boot_show = BOOTSTRAP.copy()


for c in [

    "observed_daily_alpha",

    "ci_low",

    "ci_high",

    "prob_alpha_positive"
]:

    if c in boot_show:

        boot_show[
            c
        ] = (

            boot_show[
                c
            ]

            *

            100
        )


print(
    boot_show.to_string(
        index=False
    )
)



print(
    "\n"
    +
    "=" * 100
)


print(
    "REINTEGRATION DECISION TABLE"
)


print(
    "=" * 100
)



for _, row in DECISION_TABLE.iterrows():

    print(
        "\nCandidate:",
        row[
            "candidate"
        ]
    )


    print(
        "Development wins:",
        row[
            "dev_wins"
        ],
        "/ 5"
    )


    print(
        "2026 confirmation wins:",
        row[
            "confirm_wins"
        ],
        "/ 4"
    )


    print(
        "2x cost PF:",
        row[
            "cost_2x_pf"
        ]
    )


    print(
        "Bootstrap P(alpha>0):",
        row[
            "bootstrap_p"
        ]
    )


    print(
        "Development checks:",
        row.get(
            "dev_checks"
        )
    )


    print(
        "Confirmation checks:",
        row.get(
            "confirm_checks"
        )
    )


    print(
        "PASS:",
        row[
            "pass"
        ]
    )



print(
    "\n"
    +
    "=" * 100
)


print(
    "FINAL DECISION"
)


print(
    "=" * 100
)


print(
    FINAL_DECISION
)



if FINAL_DECISION == (
    "PROMOTE_CHAMPION_PLUS_VOL_REGIME"
):

    print(

        "\n判定:"
        "\n正式30-feature HGB Championに"
        "\nVolatility + Regime を"
        "\n統合する根拠が確認されました。"
    )


elif FINAL_DECISION == (
    "PROMOTE_CHAMPION_PLUS_REGIME"
):

    print(

        "\n判定:"
        "\nRegime単独追加が最も堅牢。"
        "\nVolatilityは正式統合しません。"
    )


else:

    print(

        "\n判定:"
        "\n追加特徴量の優位性は"
        "\n正式Championを置き換えるほど"
        "\n明確ではありません。"
        "\n現行30-feature Championを維持します。"
    )



print(

    "\n注意:"
    "\n2026は既に何度も結果を確認しているため"
    "\npristine holdoutではありません。"
    "\n特徴量仕様を固定した後は"
    "\n新規forward dataで最終確認します。"
)



# ============================================================
# 18. 後で再利用するNotebook変数
# ============================================================

CHAMPION_REINTEGRATION_ANNUAL = (
    ANNUAL.copy()
)


CHAMPION_REINTEGRATION_DEV = (
    DEV_SUMMARY.copy()
)


CHAMPION_REINTEGRATION_CONFIRMATION = (
    CONFIRMATION.copy()
)


CHAMPION_REINTEGRATION_COST = (
    COST_STRESS.copy()
)


CHAMPION_REINTEGRATION_BOOTSTRAP = (
    BOOTSTRAP.copy()
)


CHAMPION_REINTEGRATION_DECISION = (
    DECISION_TABLE.copy()
)


CHAMPION_REINTEGRATION_TRADES = (
    TRADES.copy()
)


CHAMPION_REINTEGRATION_FINAL = (
    FINAL_DECISION
)



print(
    "\n検証終了"
)


print(
    "\n保存されたNotebook変数:"
)


print(
    "CHAMPION_REINTEGRATION_ANNUAL"
)


print(
    "CHAMPION_REINTEGRATION_DEV"
)


print(
    "CHAMPION_REINTEGRATION_CONFIRMATION"
)


print(
    "CHAMPION_REINTEGRATION_COST"
)


print(
    "CHAMPION_REINTEGRATION_BOOTSTRAP"
)


print(
    "CHAMPION_REINTEGRATION_DECISION"
)


print(
    "CHAMPION_REINTEGRATION_TRADES"
)


print(
    "CHAMPION_REINTEGRATION_FINAL"
)


## 元セルindex 57


In [ ]:
# ============================================================
# 15分足 OHLC SAFE REPAIR
# irregular timestamp / 5min / offset bars -> exact 15min OHLC
# ============================================================

import numpy as np
import pandas as pd


def make_safe_15m_ohlc(frame):

    if not isinstance(frame, pd.DataFrame):
        raise TypeError("bars が pandas.DataFrame ではありません。")

    x = frame.copy()

    # 列名標準化
    x.columns = [
        str(c).strip().lower()
        for c in x.columns
    ]

    required = {"open", "high", "low", "close"}
    missing = required - set(x.columns)

    if missing:
        raise ValueError(f"OHLC列がありません: {sorted(missing)}")

    # timestampをDatetimeIndexへ
    if not isinstance(x.index, pd.DatetimeIndex):

        time_col = next(
            (
                c for c in
                ["timestamp", "datetime", "date", "time"]
                if c in x.columns
            ),
            None
        )

        if time_col is None:
            raise ValueError(
                "DatetimeIndex または timestamp列が必要です。"
            )

        x.index = pd.to_datetime(
            x.pop(time_col),
            utc=True,
            errors="coerce"
        )

    else:
        x.index = pd.to_datetime(
            x.index,
            utc=True,
            errors="coerce"
        )

    x = x.loc[~x.index.isna()].copy()

    x = x[
        ["open", "high", "low", "close"]
    ].copy()

    for col in ["open", "high", "low", "close"]:
        x[col] = pd.to_numeric(
            x[col],
            errors="coerce"
        )

    x = x.dropna().sort_index()

    if x.empty:
        raise ValueError("有効なOHLCデータがありません。")

    # 同一timestamp重複にも対応
    if x.index.has_duplicates:

        print("[REPAIR] duplicate timestamps detected.")

        x = (
            x.groupby(level=0)
            .agg({
                "open": "first",
                "high": "max",
                "low": "min",
                "close": "last"
            })
            .sort_index()
        )

    # 元データ確認
    if len(x) >= 2:

        diffs = pd.Series(
            x.index[1:] - x.index[:-1]
        )

        positive = diffs[
            diffs > pd.Timedelta(0)
        ]

        if len(positive):
            print(
                "[BEFORE] Median interval:",
                positive.median()
            )

    print("[BEFORE] rows:", f"{len(x):,}")
    print(
        "[BEFORE] period:",
        x.index.min(),
        "->",
        x.index.max()
    )

    # --------------------------------------------------------
    # 正式な15分OHLCへ集約
    # open=最初 / high=最大 / low=最小 / close=最後
    # --------------------------------------------------------

    x15 = (
        x.resample(
            "15min",
            origin="epoch",
            label="left",
            closed="left"
        )
        .agg({
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last"
        })
    )

    # データの無い区間は補間せず削除
    x15 = x15.dropna(
        subset=["open", "high", "low", "close"]
    )

    # 非有限値除去
    finite_mask = np.isfinite(
        x15[
            ["open", "high", "low", "close"]
        ].to_numpy(dtype=float)
    ).all(axis=1)

    x15 = x15.loc[finite_mask].copy()

    # 0以下除去
    positive_mask = (
        x15[
            ["open", "high", "low", "close"]
        ] > 0
    ).all(axis=1)

    x15 = x15.loc[positive_mask].copy()

    # OHLC整合性
    high_required = x15[
        ["open", "close", "low"]
    ].max(axis=1)

    low_required = x15[
        ["open", "close", "high"]
    ].min(axis=1)

    valid = (
        (x15["high"] >= high_required)
        &
        (x15["low"] <= low_required)
    )

    x15 = x15.loc[valid].copy()

    # 15分グリッド最終確認
    idx = x15.index

    grid_ok = (
        (idx.minute % 15 == 0)
        &
        (idx.second == 0)
        &
        (idx.microsecond == 0)
    )

    if not np.asarray(grid_ok).all():
        raise RuntimeError("15分足への変換に失敗しました。")

    print()
    print("=" * 70)
    print("15-MINUTE REPAIR COMPLETE")
    print("=" * 70)

    print("[AFTER] rows:", f"{len(x15):,}")
    print(
        "[AFTER] period:",
        x15.index.min(),
        "->",
        x15.index.max()
    )

    if len(x15) >= 2:

        diffs15 = pd.Series(
            x15.index[1:] - x15.index[:-1]
        )

        positive15 = diffs15[
            diffs15 > pd.Timedelta(0)
        ]

        if len(positive15):
            print(
                "[AFTER] Median interval:",
                positive15.median()
            )

    bad_grid_count = int(
        (
            (x15.index.minute % 15 != 0)
            |
            (x15.index.second != 0)
            |
            (x15.index.microsecond != 0)
        ).sum()
    )

    print(
        "[AFTER] Bad 15m timestamps:",
        bad_grid_count
    )

    print()
    print(x15.head())

    return x15


# ============================================================
# 実行
# ============================================================

try:
    bars
except NameError:
    raise RuntimeError(
        "Notebook内に `bars` がありません。"
        "先に元OHLCデータを読み込んでください。"
    )

bars = make_safe_15m_ohlc(bars)

# 別名にも対応
bars15 = bars.copy()
bars_15m = bars.copy()

print()
print("=" * 70)
print("READY")
print("=" * 70)
print(f"bars = {len(bars):,} rows")


## 元セルindex 58


In [ ]:
# ============================================================
# CHAMPION REINTEGRATION TEST
# HGB + BASE + VOLATILITY + REGIME
#
# 目的:
# 1. 修復済み bars をそのまま使用
# 2. BASE 30特徴量 vs FORMAL CHAMPION 50特徴量
# 3. 完全時系列 Nested Walk-Forward
# 4. Calibration -> Threshold/Session -> Position Sizing
# 5. 30分固定Exit
# 6. 2020-2025 Development OOS
# 7. 2026 Confirmation
# 8. Cost 1x / 1.5x / 2x Stress
# ============================================================

import math
import warnings
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

warnings.filterwarnings("ignore")

# ============================================================
# 0. 固定設定
# ============================================================

RANDOM_STATE = 42

COST = 0.00004

DEVELOPMENT_YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
CONFIRMATION_YEAR = 2026

THRESHOLDS = [0.55, 0.56, 0.58, 0.60, 0.62, 0.65]

SESSIONS = [
    "ALL",
    "UTC_13_24",
    "UTC_21_24",
    "EXCLUDE_08_13",
]

SIZING_POLICIES = [
    "FIXED",
    "GENTLE",
    "MODERATE",
    "STRONG",
]

CALIBRATION_METHODS = [
    "RAW",
    "PLATT",
    "ISOTONIC",
]

HGB_CONFIG = {
    "learning_rate": 0.05,
    "max_iter": 250,
    "max_leaf_nodes": 15,
    "min_samples_leaf": 30,
    "l2_regularization": 1.0,
    "early_stopping": False,
    "random_state": RANDOM_STATE,
}

MIN_TRAIN_ROWS = 10000
MIN_EVAL_ROWS = 500
MIN_VALIDATION_TRADES = 40


# ============================================================
# 1. bars確認
# ============================================================

if "bars" not in globals():
    raise RuntimeError(
        "bars がNotebookにありません。\n"
        "直前の15分足修復セルを先に実行してください。"
    )

if not isinstance(bars, pd.DataFrame):
    raise TypeError("bars は pandas.DataFrame である必要があります。")

BARS = bars.copy()

# OHLC列を小文字へ
BARS.columns = [str(c).strip().lower() for c in BARS.columns]

required_ohlc = ["open", "high", "low", "close"]

missing = [c for c in required_ohlc if c not in BARS.columns]
if missing:
    raise ValueError(f"OHLC列が不足しています: {missing}")

BARS = BARS[required_ohlc].copy()

# indexをDatetimeIndexへ
if not isinstance(BARS.index, pd.DatetimeIndex):
    raise TypeError("bars.index が DatetimeIndex ではありません。")

if BARS.index.tz is None:
    BARS.index = BARS.index.tz_localize("UTC")
else:
    BARS.index = BARS.index.tz_convert("UTC")

BARS = BARS.sort_index()

# 重複を除去
BARS = BARS[~BARS.index.duplicated(keep="first")]

# 数値化
for c in required_ohlc:
    BARS[c] = pd.to_numeric(BARS[c], errors="coerce")

BARS = BARS.dropna()

# 15分グリッド確認
idx = BARS.index

bad_grid = (
    (idx.minute % 15 != 0)
    | (idx.second != 0)
    | (idx.microsecond != 0)
)

if np.asarray(bad_grid).any():
    examples = idx[np.asarray(bad_grid)][:10]
    raise ValueError(
        "15分グリッド外timestampがまだ残っています。\n"
        f"例: {list(examples)}"
    )

print("=" * 80)
print("DATA CHECK")
print("=" * 80)
print(f"Rows: {len(BARS):,}")
print(f"Period: {BARS.index.min()} -> {BARS.index.max()}")
print(f"Bad 15m timestamps: {np.asarray(bad_grid).sum()}")


# ============================================================
# 2. 特徴量
# ============================================================

BASE_FEATURES = [
    "return_1",
    "return_2",
    "return_4",
    "return_8",
    "return_16",

    "vol_4",
    "vol_8",
    "vol_16",
    "vol_32",

    "ma5_distance",
    "ma5_slope",
    "ma10_distance",
    "ma10_slope",
    "ma20_distance",
    "ma20_slope",
    "ma50_distance",
    "ma50_slope",
    "ma100_distance",
    "ma100_slope",

    "body",
    "upper_wick",
    "lower_wick",
    "range_pct",

    "rsi14",
    "atr14",

    "distance_high_16",
    "distance_low_16",

    "hour_sin",
    "hour_cos",
    "weekday",
]


REGIME_ADDITIONS = [
    "adx14",
    "plus_di14",
    "minus_di14",

    "ma20_vs_ma50",
    "ma50_vs_ma100",

    "trend_strength_20",
    "trend_strength_50",

    "breakout_pos_32",
    "range_position_64",

    "vol_ratio_8_32",
    "atr_ratio_7_28",
]


VOLATILITY_ADDITIONS = [
    "vol_2",
    "vol_6",
    "vol_12",
    "vol_24",

    "atr7",
    "atr28",

    "range_mean_4",
    "range_mean_16",
    "range_std_16",

    # Regimeと共通
    "vol_ratio_8_32",
    "atr_ratio_7_28",
    "breakout_pos_32",
    "trend_strength_20",
]


CHAMPION_FEATURES = list(
    dict.fromkeys(
        BASE_FEATURES
        + REGIME_ADDITIONS
        + VOLATILITY_ADDITIONS
    )
)

print("\nFeature configuration")
print("BASE:", len(BASE_FEATURES))
print("Regime additions:", len(REGIME_ADDITIONS))
print("Volatility additions:", len(VOLATILITY_ADDITIONS))
print("FORMAL CHAMPION:", len(CHAMPION_FEATURES))

if len(BASE_FEATURES) != 30:
    raise AssertionError("BASE features should be 30.")

if len(CHAMPION_FEATURES) != 50:
    raise AssertionError(
        f"Champion features should be 50, got {len(CHAMPION_FEATURES)}"
    )


# ============================================================
# 3. RSI
# ============================================================

def calc_rsi(close, period=14):

    delta = close.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)

    return 100 - (100 / (1 + rs))


# ============================================================
# 4. 全特徴量作成
# ============================================================

def make_all_features(b):

    x = b.copy()

    # ------------------------
    # Returns
    # ------------------------

    for n in [1, 2, 4, 8, 16]:
        x[f"return_{n}"] = x["close"].pct_change(n)

    # ------------------------
    # BASE volatility
    # ------------------------

    for n in [4, 8, 16, 32]:
        x[f"vol_{n}"] = x["return_1"].rolling(n).std()

    # Champion volatility
    for n in [2, 6, 12, 24]:
        x[f"vol_{n}"] = x["return_1"].rolling(n).std()

    # ------------------------
    # Moving averages
    # ------------------------

    ma = {}

    for p in [5, 10, 20, 50, 100]:

        ma[p] = x["close"].rolling(p).mean()

        x[f"ma{p}_distance"] = (
            x["close"] / ma[p] - 1
        )

        x[f"ma{p}_slope"] = (
            ma[p].pct_change()
        )

    # ------------------------
    # Candle
    # ------------------------

    candle_range = (
        x["high"] - x["low"]
    ).replace(0, np.nan)

    x["body"] = (
        x["close"] - x["open"]
    ) / candle_range

    x["upper_wick"] = (
        x["high"]
        - x[["open", "close"]].max(axis=1)
    ) / candle_range

    x["lower_wick"] = (
        x[["open", "close"]].min(axis=1)
        - x["low"]
    ) / candle_range

    x["range_pct"] = (
        x["high"] - x["low"]
    ) / x["close"]

    # ------------------------
    # RSI
    # ------------------------

    x["rsi14"] = calc_rsi(
        x["close"],
        14
    ) / 100.0

    # ------------------------
    # True Range
    # ------------------------

    prev_close = x["close"].shift(1)

    tr = pd.concat(
        [
            x["high"] - x["low"],
            (x["high"] - prev_close).abs(),
            (x["low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)

    atr14_abs = tr.rolling(14).mean()

    x["atr14"] = (
        atr14_abs / x["close"]
    )

    # extra ATR
    atr7_abs = tr.rolling(7).mean()
    atr28_abs = tr.rolling(28).mean()

    x["atr7"] = (
        atr7_abs / x["close"]
    )

    x["atr28"] = (
        atr28_abs / x["close"]
    )

    x["atr_ratio_7_28"] = (
        atr7_abs
        / atr28_abs.replace(0, np.nan)
    )

    # ------------------------
    # Recent high / low
    # ------------------------

    high16 = x["high"].rolling(16).max()
    low16 = x["low"].rolling(16).min()

    x["distance_high_16"] = (
        high16 - x["close"]
    ) / x["close"]

    x["distance_low_16"] = (
        x["close"] - low16
    ) / x["close"]

    # ------------------------
    # Time
    # ------------------------

    hour = (
        x.index.hour
        + x.index.minute / 60.0
    )

    x["hour_sin"] = np.sin(
        2 * np.pi * hour / 24
    )

    x["hour_cos"] = np.cos(
        2 * np.pi * hour / 24
    )

    x["weekday"] = (
        x.index.dayofweek / 4.0
    )

    # ========================================================
    # REGIME
    # ========================================================

    x["ma20_vs_ma50"] = (
        ma[20] / ma[50] - 1
    )

    x["ma50_vs_ma100"] = (
        ma[50] / ma[100] - 1
    )

    x["trend_strength_20"] = (
        ma[20] / ma[20].shift(4) - 1
    ).abs()

    x["trend_strength_50"] = (
        ma[50] / ma[50].shift(4) - 1
    ).abs()

    high32 = x["high"].rolling(32).max()
    low32 = x["low"].rolling(32).min()

    width32 = (
        high32 - low32
    ).replace(0, np.nan)

    x["breakout_pos_32"] = (
        x["close"] - low32
    ) / width32

    high64 = x["high"].rolling(64).max()
    low64 = x["low"].rolling(64).min()

    width64 = (
        high64 - low64
    ).replace(0, np.nan)

    x["range_position_64"] = (
        x["close"] - low64
    ) / width64

    x["vol_ratio_8_32"] = (
        x["vol_8"]
        / x["vol_32"].replace(0, np.nan)
    )

    # ------------------------
    # ADX / DI
    # ------------------------

    up_move = x["high"].diff()
    down_move = -x["low"].diff()

    plus_dm = pd.Series(
        np.where(
            (up_move > down_move) & (up_move > 0),
            up_move,
            0.0
        ),
        index=x.index
    )

    minus_dm = pd.Series(
        np.where(
            (down_move > up_move) & (down_move > 0),
            down_move,
            0.0
        ),
        index=x.index
    )

    atr_wilder = tr.ewm(
        alpha=1/14,
        adjust=False,
        min_periods=14
    ).mean()

    plus_di = (
        100
        * plus_dm.ewm(
            alpha=1/14,
            adjust=False,
            min_periods=14
        ).mean()
        / atr_wilder.replace(0, np.nan)
    )

    minus_di = (
        100
        * minus_dm.ewm(
            alpha=1/14,
            adjust=False,
            min_periods=14
        ).mean()
        / atr_wilder.replace(0, np.nan)
    )

    dx = (
        100
        * (plus_di - minus_di).abs()
        / (plus_di + minus_di).replace(0, np.nan)
    )

    adx = dx.ewm(
        alpha=1/14,
        adjust=False,
        min_periods=14
    ).mean()

    # 0-1へ
    x["plus_di14"] = plus_di / 100.0
    x["minus_di14"] = minus_di / 100.0
    x["adx14"] = adx / 100.0

    # ========================================================
    # VOLATILITY
    # ========================================================

    x["range_mean_4"] = (
        x["range_pct"].rolling(4).mean()
    )

    x["range_mean_16"] = (
        x["range_pct"].rolling(16).mean()
    )

    x["range_std_16"] = (
        x["range_pct"].rolling(16).std()
    )

    return x.replace(
        [np.inf, -np.inf],
        np.nan
    )


# ============================================================
# 5. 30分固定ラベル
#
# signal t
# entry Open(t+1)
# exit  Close(t+2)
# ============================================================

def prepare_dataset(b, features):

    x = make_all_features(b)

    times = pd.Series(
        b.index,
        index=b.index
    )

    x["entry_time"] = times.shift(-1)

    x["label_end"] = (
        times.shift(-2)
        + pd.Timedelta(minutes=15)
    )

    x["entry_price"] = (
        b["open"].shift(-1)
    )

    x["exit_price"] = (
        b["close"].shift(-2)
    )

    x["future_return"] = (
        x["exit_price"]
        / x["entry_price"]
        - 1
    )

    x["target"] = (
        x["future_return"] > 0
    ).astype(int)

    # t -> t+1 -> t+2 が連続15分足か確認
    continuous = (
        (times.shift(-1) - times)
        .eq(pd.Timedelta(minutes=15))
        &
        (times.shift(-2) - times)
        .eq(pd.Timedelta(minutes=30))
    )

    required = (
        list(features)
        + [
            "entry_time",
            "label_end",
            "entry_price",
            "exit_price",
            "future_return",
            "target",
        ]
    )

    out = (
        x.loc[continuous]
        .dropna(subset=required)
        .copy()
    )

    return out


# ============================================================
# 6. Stats
# ============================================================

def stats_of_returns(r):

    r = np.asarray(r, dtype=float)
    r = r[np.isfinite(r)]

    if len(r) == 0:

        return {
            "trades": 0,
            "win_rate": np.nan,
            "avg_return": np.nan,
            "profit_factor": np.nan,
            "growth": 0.0,
            "max_dd": np.nan,
            "return_to_dd": np.nan,
        }

    gains = r[r > 0].sum()
    losses = -r[r < 0].sum()

    if losses > 0:
        pf = gains / losses

    elif gains > 0:
        pf = np.inf

    else:
        pf = np.nan

    equity = np.r_[
        1.0,
        np.cumprod(1 + r)
    ]

    peak = np.maximum.accumulate(equity)

    dd = (
        equity / peak - 1
    )

    max_dd = float(dd.min())
    growth = float(equity[-1] - 1)

    if (
        np.isfinite(max_dd)
        and max_dd < 0
    ):
        rdd = growth / abs(max_dd)

    else:
        rdd = np.nan

    return {
        "trades": int(len(r)),
        "win_rate": float((r > 0).mean()),
        "avg_return": float(r.mean()),
        "profit_factor": float(pf),
        "growth": growth,
        "max_dd": max_dd,
        "return_to_dd": float(rdd),
    }


# ============================================================
# 7. Annual split
# ============================================================

def make_split(data, test_year):

    validation_year = test_year - 1

    val_start = pd.Timestamp(
        f"{validation_year}-01-01",
        tz="UTC"
    )

    test_start = pd.Timestamp(
        f"{test_year}-01-01",
        tz="UTC"
    )

    test_end = pd.Timestamp(
        f"{test_year + 1}-01-01",
        tz="UTC"
    )

    train = data.loc[
        (data.index < val_start)
        &
        (data["label_end"] <= val_start)
    ].copy()

    validation = data.loc[
        (data.index >= val_start)
        &
        (data.index < test_start)
        &
        (data["label_end"] <= test_start)
    ].copy()

    final_train = data.loc[
        (data.index < test_start)
        &
        (data["label_end"] <= test_start)
    ].copy()

    test = data.loc[
        (data.index >= test_start)
        &
        (data.index < test_end)
        &
        (data["label_end"] <= test_end)
    ].copy()

    if (
        len(train) < MIN_TRAIN_ROWS
        or len(validation) < MIN_EVAL_ROWS
        or len(test) < MIN_EVAL_ROWS
    ):
        return None

    return {
        "test_year": test_year,
        "validation_year": validation_year,
        "train": train,
        "validation": validation,
        "final_train": final_train,
        "test": test,
    }


# ============================================================
# 8. Model
# ============================================================

def fit_hgb(data, features):

    model = HistGradientBoostingClassifier(
        **HGB_CONFIG
    )

    model.fit(
        data[features],
        data["target"]
    )

    return model


def raw_probability(model, data, features):

    return model.predict_proba(
        data[features]
    )[:, 1]


# ============================================================
# 9. Calibration
# ============================================================

class RawCalibrator:

    def fit(self, p, y):
        return self

    def predict(self, p):
        return np.asarray(p)


class PlattCalibrator:

    def __init__(self):

        self.model = LogisticRegression(
            solver="lbfgs",
            random_state=RANDOM_STATE
        )

    def fit(self, p, y):

        self.model.fit(
            np.asarray(p).reshape(-1, 1),
            y
        )

        return self

    def predict(self, p):

        return self.model.predict_proba(
            np.asarray(p).reshape(-1, 1)
        )[:, 1]


class IsotonicCalibrator:

    def __init__(self):

        self.model = IsotonicRegression(
            y_min=0,
            y_max=1,
            out_of_bounds="clip"
        )

    def fit(self, p, y):

        self.model.fit(
            np.asarray(p),
            y
        )

        return self

    def predict(self, p):

        return np.asarray(
            self.model.predict(
                np.asarray(p)
            )
        )


def expanding_oof(train, features):

    years = sorted(
        train.index.year.unique()
    )

    parts = []

    for y in years:

        prior = [
            yy
            for yy in years
            if yy < y
        ]

        if len(prior) < 2:
            continue

        start = pd.Timestamp(
            f"{y}-01-01",
            tz="UTC"
        )

        end = pd.Timestamp(
            f"{y+1}-01-01",
            tz="UTC"
        )

        hist = train.loc[
            (train.index < start)
            &
            (train["label_end"] <= start)
        ]

        oof = train.loc[
            (train.index >= start)
            &
            (train.index < end)
            &
            (train["label_end"] <= end)
        ]

        if (
            len(hist) < MIN_TRAIN_ROWS
            or len(oof) < MIN_EVAL_ROWS
        ):
            continue

        if hist["target"].nunique() < 2:
            continue

        model = fit_hgb(
            hist,
            features
        )

        p = raw_probability(
            model,
            oof,
            features
        )

        parts.append(
            pd.DataFrame(
                {
                    "prob": p,
                    "target": oof["target"].values
                },
                index=oof.index
            )
        )

    if not parts:

        return pd.DataFrame(
            columns=["prob", "target"]
        )

    return pd.concat(
        parts
    ).sort_index()


def fit_calibrator(method, oof):

    if (
        method == "RAW"
        or len(oof) < 500
    ):
        return RawCalibrator()

    p = oof["prob"].values
    y = oof["target"].values

    if method == "PLATT":

        return (
            PlattCalibrator()
            .fit(p, y)
        )

    if method == "ISOTONIC":

        if len(oof) < 1000:
            return RawCalibrator()

        return (
            IsotonicCalibrator()
            .fit(p, y)
        )

    return RawCalibrator()


def choose_calibration(
    train,
    validation,
    features
):

    oof = expanding_oof(
        train,
        features
    )

    model = fit_hgb(
        train,
        features
    )

    raw_val = raw_probability(
        model,
        validation,
        features
    )

    rows = []

    for method in CALIBRATION_METHODS:

        cal = fit_calibrator(
            method,
            oof
        )

        p = np.clip(
            cal.predict(raw_val),
            0,
            1
        )

        brier = brier_score_loss(
            validation["target"],
            p
        )

        rows.append(
            {
                "method": method,
                "brier": brier
            }
        )

    table = (
        pd.DataFrame(rows)
        .sort_values(
            ["brier", "method"]
        )
    )

    chosen = table.iloc[0]["method"]

    return chosen, raw_val


# ============================================================
# 10. Prediction frame
# ============================================================

def prediction_frame(
    data,
    calibrated_probability
):

    p = np.asarray(
        calibrated_probability
    )

    direction_sign = np.where(
        p >= 0.5,
        1.0,
        -1.0
    )

    out = data[
        [
            "entry_time",
            "label_end",
            "future_return",
            "target"
        ]
    ].copy()

    out["p_up"] = p

    out["confidence"] = np.maximum(
        p,
        1 - p
    )

    out["gross_return"] = (
        data["future_return"].values
        * direction_sign
    )

    return out


# ============================================================
# 11. Session
# ============================================================

def session_mask(index, policy):

    hour = index.hour

    if policy == "ALL":

        return np.ones(
            len(index),
            dtype=bool
        )

    if policy == "UTC_13_24":

        return (
            (hour >= 13)
            & (hour < 24)
        )

    if policy == "UTC_21_24":

        return (
            (hour >= 21)
            & (hour < 24)
        )

    if policy == "EXCLUDE_08_13":

        return ~(
            (hour >= 8)
            & (hour < 13)
        )

    raise ValueError(policy)


# ============================================================
# 12. Trade selection
# ============================================================

def select_trades(
    pred,
    threshold,
    session
):

    mask = (
        (pred["confidence"] >= threshold)
        &
        session_mask(
            pred.index,
            session
        )
    )

    candidates = (
        pred.loc[mask]
        .sort_index()
    )

    selected = []
    next_free = None

    for row in candidates.itertuples():

        if (
            next_free is not None
            and row.entry_time < next_free
        ):
            continue

        selected.append(
            row.Index
        )

        next_free = row.label_end

    trades = candidates.loc[
        selected
    ].copy()

    trades["base_net_return"] = (
        trades["gross_return"]
        - COST
    )

    return trades


# ============================================================
# 13. Threshold + Session
# ============================================================

def choose_threshold_session(
    validation_predictions
):

    rows = []

    best = None
    best_key = None

    for threshold in THRESHOLDS:

        for session in SESSIONS:

            trades = select_trades(
                validation_predictions,
                threshold,
                session
            )

            s = stats_of_returns(
                trades["base_net_return"]
            )

            eligible = (
                s["trades"]
                >= MIN_VALIDATION_TRADES
            )

            if eligible:

                score = (
                    s["avg_return"]
                    * math.sqrt(
                        s["trades"]
                    )
                )

            else:

                score = np.nan

            rows.append(
                {
                    "threshold": threshold,
                    "session": session,
                    "eligible": eligible,
                    "score": score,
                    **s
                }
            )

            if not eligible:
                continue

            key = (
                score,
                (
                    s["profit_factor"]
                    if np.isfinite(
                        s["profit_factor"]
                    )
                    else -999
                ),
                s["trades"],
                -threshold
            )

            if (
                best_key is None
                or key > best_key
            ):

                best_key = key
                best = (
                    threshold,
                    session
                )

    if best is None:

        raise RuntimeError(
            "Validationで十分な取引数を持つ"
            "Threshold/Sessionがありません。"
        )

    return (
        best[0],
        best[1],
        pd.DataFrame(rows)
    )


# ============================================================
# 14. Position sizing
# ============================================================

def raw_size(
    confidence,
    threshold,
    policy
):

    confidence = np.asarray(
        confidence
    )

    edge = (
        confidence - threshold
    ) / max(
        1 - threshold,
        1e-8
    )

    edge = np.clip(
        edge,
        0,
        1
    )

    if policy == "FIXED":
        return np.ones_like(edge)

    if policy == "GENTLE":
        return 0.85 + 0.30 * edge

    if policy == "MODERATE":
        return 0.70 + 0.60 * edge

    if policy == "STRONG":
        return 0.50 + 1.00 * edge

    raise ValueError(policy)


def choose_sizing(
    trades,
    threshold
):

    if trades.empty:

        return (
            "FIXED",
            1.0
        )

    rows = []

    for policy in SIZING_POLICIES:

        raw = raw_size(
            trades["confidence"],
            threshold,
            policy
        )

        # 平均Exposureを1へ
        scale = (
            1.0 / raw.mean()
        )

        size = (
            raw * scale
        )

        r = (
            size
            * (
                trades["gross_return"].values
                - COST
            )
        )

        s = stats_of_returns(r)

        rows.append(
            {
                "policy": policy,
                "scale": scale,
                **s
            }
        )

    table = pd.DataFrame(rows)

    fixed = table.loc[
        table["policy"] == "FIXED"
    ].iloc[0]

    candidates = []

    for _, row in table.iterrows():

        if row["policy"] == "FIXED":
            continue

        if (
            row["avg_return"]
            >= fixed["avg_return"]
            and
            row["profit_factor"]
            >= fixed["profit_factor"]
            and
            row["return_to_dd"]
            >= fixed["return_to_dd"]
        ):

            candidates.append(row)

    if not candidates:

        return (
            "FIXED",
            1.0
        )

    chosen = max(
        candidates,
        key=lambda r: (
            r["return_to_dd"],
            r["profit_factor"],
            r["avg_return"]
        )
    )

    return (
        chosen["policy"],
        float(chosen["scale"])
    )


def apply_sizing(
    trades,
    threshold,
    policy,
    scale,
    cost_multiplier=1.0
):

    out = trades.copy()

    size = (
        raw_size(
            out["confidence"],
            threshold,
            policy
        )
        * scale
    )

    size = np.clip(
        size,
        0.25,
        2.0
    )

    out["position_size"] = size

    out["net_return"] = (
        size
        * (
            out["gross_return"]
            - COST * cost_multiplier
        )
    )

    return out


# ============================================================
# 15. 1年評価
# ============================================================

def evaluate_year(
    data,
    features,
    test_year,
    feature_name
):

    split = make_split(
        data,
        test_year
    )

    if split is None:
        return None

    train = split["train"]
    validation = split["validation"]
    final_train = split["final_train"]
    test = split["test"]

    # --------------------------------
    # Calibration selection
    # --------------------------------

    calibration, raw_val = (
        choose_calibration(
            train,
            validation,
            features
        )
    )

    train_oof = expanding_oof(
        train,
        features
    )

    val_calibrator = fit_calibrator(
        calibration,
        train_oof
    )

    p_val = np.clip(
        val_calibrator.predict(
            raw_val
        ),
        0,
        1
    )

    val_pred = prediction_frame(
        validation,
        p_val
    )

    # --------------------------------
    # Threshold / Session
    # --------------------------------

    threshold, session, _ = (
        choose_threshold_session(
            val_pred
        )
    )

    val_selected = select_trades(
        val_pred,
        threshold,
        session
    )

    # --------------------------------
    # Position sizing
    # --------------------------------

    sizing_policy, scale = (
        choose_sizing(
            val_selected,
            threshold
        )
    )

    # --------------------------------
    # Test前に全historyで再fit
    # --------------------------------

    final_oof = expanding_oof(
        final_train,
        features
    )

    final_calibrator = fit_calibrator(
        calibration,
        final_oof
    )

    final_model = fit_hgb(
        final_train,
        features
    )

    raw_test = raw_probability(
        final_model,
        test,
        features
    )

    p_test = np.clip(
        final_calibrator.predict(
            raw_test
        ),
        0,
        1
    )

    test_pred = prediction_frame(
        test,
        p_test
    )

    selected = select_trades(
        test_pred,
        threshold,
        session
    )

    final_trades = apply_sizing(
        selected,
        threshold,
        sizing_policy,
        scale,
        cost_multiplier=1.0
    )

    s = stats_of_returns(
        final_trades["net_return"]
    )

    if test["target"].nunique() == 2:

        auc = roc_auc_score(
            test["target"],
            p_test
        )

    else:

        auc = np.nan

    result = {
        "feature_set": feature_name,
        "test_year": test_year,
        "validation_year": test_year - 1,
        "calibration": calibration,
        "threshold": threshold,
        "session": session,
        "sizing_policy": sizing_policy,
        "auc": auc,
        **s
    }

    final_trades = (
        final_trades.copy()
    )

    final_trades[
        "feature_set"
    ] = feature_name

    final_trades[
        "test_year"
    ] = test_year

    return (
        result,
        final_trades
    )


# ============================================================
# 16. 全年評価
# ============================================================

FEATURE_SETS = {
    "BASE": BASE_FEATURES,
    "CHAMPION": CHAMPION_FEATURES,
}

all_results = []
all_trades = []

for feature_name, features in FEATURE_SETS.items():

    print("\n")
    print("=" * 80)
    print(feature_name)
    print("=" * 80)

    data = prepare_dataset(
        BARS,
        features
    )

    print(
        f"Usable rows: {len(data):,}"
    )

    for year in (
        DEVELOPMENT_YEARS
        + [CONFIRMATION_YEAR]
    ):

        print(
            f"\nTEST YEAR {year}"
        )

        result = evaluate_year(
            data,
            features,
            year,
            feature_name
        )

        if result is None:

            print("SKIPPED")
            continue

        row, trades = result

        all_results.append(row)
        all_trades.append(trades)

        print(
            "Calibration:",
            row["calibration"]
        )

        print(
            "Threshold:",
            row["threshold"]
        )

        print(
            "Session:",
            row["session"]
        )

        print(
            "Sizing:",
            row["sizing_policy"]
        )

        print(
            f"AUC: {row['auc']:.4f}"
        )

        print(
            f"Trades: {row['trades']}"
        )

        print(
            f"Win: {row['win_rate']*100:.2f}%"
        )

        print(
            f"Avg Return: "
            f"{row['avg_return']*100:.5f}%"
        )

        print(
            f"PF: "
            f"{row['profit_factor']:.3f}"
        )

        print(
            f"Growth: "
            f"{row['growth']*100:.3f}%"
        )

        print(
            f"Max DD: "
            f"{row['max_dd']*100:.3f}%"
        )


ANNUAL = pd.DataFrame(
    all_results
)

TRADES = pd.concat(
    all_trades
).sort_index()


# ============================================================
# 17. Development summary
# ============================================================

summary_rows = []

for feature_name in FEATURE_SETS:

    sub = ANNUAL.loc[
        (ANNUAL["feature_set"] == feature_name)
        &
        (ANNUAL["test_year"].isin(
            DEVELOPMENT_YEARS
        ))
    ]

    t = TRADES.loc[
        (TRADES["feature_set"] == feature_name)
        &
        (TRADES["test_year"].isin(
            DEVELOPMENT_YEARS
        ))
    ]

    s = stats_of_returns(
        t["net_return"]
    )

    summary_rows.append(
        {
            "feature_set": feature_name,
            "years": len(sub),
            "positive_years":
                int(
                    (
                        sub["avg_return"] > 0
                    ).sum()
                ),
            "pf_above_1_years":
                int(
                    (
                        sub["profit_factor"] > 1
                    ).sum()
                ),
            "mean_auc":
                sub["auc"].mean(),
            **s
        }
    )

SUMMARY = pd.DataFrame(
    summary_rows
)


# ============================================================
# 18. Confirmation 2026
# ============================================================

CONFIRMATION = ANNUAL.loc[
    ANNUAL["test_year"]
    == CONFIRMATION_YEAR
].copy()


# ============================================================
# 19. Cost Stress
# ============================================================

cost_rows = []

for feature_name in FEATURE_SETS:

    t = TRADES.loc[
        (TRADES["feature_set"] == feature_name)
        &
        (
            TRADES["test_year"]
            .isin(DEVELOPMENT_YEARS)
        )
    ]

    for mult in [1.0, 1.5, 2.0]:

        r = (
            t["position_size"].values
            * (
                t["gross_return"].values
                - COST * mult
            )
        )

        s = stats_of_returns(r)

        cost_rows.append(
            {
                "feature_set": feature_name,
                "cost_x": mult,
                **s
            }
        )

COST_STRESS = pd.DataFrame(
    cost_rows
)


# ============================================================
# 20. Automatic decision
# ============================================================

base_dev = SUMMARY.loc[
    SUMMARY["feature_set"]
    == "BASE"
].iloc[0]

champ_dev = SUMMARY.loc[
    SUMMARY["feature_set"]
    == "CHAMPION"
].iloc[0]

champ_2026 = CONFIRMATION.loc[
    CONFIRMATION["feature_set"]
    == "CHAMPION"
].iloc[0]

champ_2x = COST_STRESS.loc[
    (COST_STRESS["feature_set"] == "CHAMPION")
    &
    (COST_STRESS["cost_x"] == 2.0)
].iloc[0]


dev_positive = (
    champ_dev["positive_years"]
    >= 5
)

dev_pf_years = (
    champ_dev["pf_above_1_years"]
    >= 5
)

beats_base_avg = (
    champ_dev["avg_return"]
    > base_dev["avg_return"]
)

beats_base_pf = (
    champ_dev["profit_factor"]
    > base_dev["profit_factor"]
)

beats_base_rdd = (
    champ_dev["return_to_dd"]
    > base_dev["return_to_dd"]
)

confirmation_ok = (
    champ_2026["avg_return"] > 0
    and champ_2026["profit_factor"] > 1
)

cost_ok = (
    champ_2x["profit_factor"] > 1
    and champ_2x["avg_return"] > 0
)

incremental_wins = sum(
    [
        beats_base_avg,
        beats_base_pf,
        beats_base_rdd,
    ]
)

if (
    dev_positive
    and dev_pf_years
    and incremental_wins >= 2
    and confirmation_ok
    and cost_ok
):

    DECISION = (
        "FORMAL CHAMPION PIPELINE PASSES "
        "REINTEGRATION TEST"
    )

elif (
    dev_positive
    and dev_pf_years
    and confirmation_ok
):

    DECISION = (
        "CHAMPION PIPELINE REMAINS ROBUST, "
        "BUT INCREMENTAL VALUE IS NOT FULLY CLEAR"
    )

else:

    DECISION = (
        "CHAMPION REINTEGRATION IS NOT YET ROBUST"
    )


# ============================================================
# 21. 結果表示
# ============================================================

pd.set_option(
    "display.max_columns",
    100
)

pd.set_option(
    "display.width",
    220
)

print("\n")
print("=" * 90)
print("ANNUAL RESULTS")
print("=" * 90)

print(
    ANNUAL.to_string(
        index=False
    )
)

print("\n")
print("=" * 90)
print("DEVELOPMENT OOS 2020-2025")
print("=" * 90)

print(
    SUMMARY.to_string(
        index=False
    )
)

print("\n")
print("=" * 90)
print("CONFIRMATION 2026")
print("=" * 90)

print(
    CONFIRMATION.to_string(
        index=False
    )
)

print("\n")
print("=" * 90)
print("COST STRESS")
print("=" * 90)

print(
    COST_STRESS.to_string(
        index=False
    )
)

print("\n")
print("=" * 90)
print("AUTOMATIC DIAGNOSTIC")
print("=" * 90)

print(
    "Development positive years:",
    champ_dev["positive_years"],
    "/",
    champ_dev["years"]
)

print(
    "Development PF > 1 years:",
    champ_dev["pf_above_1_years"],
    "/",
    champ_dev["years"]
)

print(
    "Champion > BASE Avg Return:",
    beats_base_avg
)

print(
    "Champion > BASE PF:",
    beats_base_pf
)

print(
    "Champion > BASE Return/DD:",
    beats_base_rdd
)

print(
    "2026 confirmation:",
    confirmation_ok
)

print(
    "2x cost survives:",
    cost_ok
)

print("\n")
print("=" * 90)
print("FINAL DECISION")
print("=" * 90)

print(DECISION)


# ============================================================
# 22. Notebook保存変数
# ============================================================

CHAMPION_REINTEGRATION_ANNUAL = (
    ANNUAL.copy()
)

CHAMPION_REINTEGRATION_TRADES = (
    TRADES.copy()
)

CHAMPION_REINTEGRATION_SUMMARY = (
    SUMMARY.copy()
)

CHAMPION_REINTEGRATION_CONFIRMATION = (
    CONFIRMATION.copy()
)

CHAMPION_REINTEGRATION_COST = (
    COST_STRESS.copy()
)

CHAMPION_REINTEGRATION_DECISION = (
    DECISION
)

print("\n検証終了")
print(
    "保存Notebook変数:"
)
print(
    "CHAMPION_REINTEGRATION_ANNUAL"
)
print(
    "CHAMPION_REINTEGRATION_TRADES"
)
print(
    "CHAMPION_REINTEGRATION_SUMMARY"
)
print(
    "CHAMPION_REINTEGRATION_CONFIRMATION"
)
print(
    "CHAMPION_REINTEGRATION_COST"
)
print(
    "CHAMPION_REINTEGRATION_DECISION"
)
